# Reproducing Varela and Wilson (2020) from CRCNS hc-24

Paper: Varela C, Wilson MA (2020). mPFC spindle cycles organize sparse thalamic activation and
recently active CA1 cells during non-REM sleep. *eLife* 9:e48881.
[doi:10.7554/elife.48881](https://doi.org/10.7554/elife.48881). The claims file was written from the
version of record (PMC7319772), with its figure supplements, Table 1 and Methods.

Data set: CRCNS hc-24, [doi:10.6080/K0K35RVG](https://doi.org/10.6080/K0K35RVG), read from the
public bucket at `crcnsarchive/hc-24/live/data.zip` (100 MB), opened in place over HTTP. CRCNS data
sets are not versioned, so the version is recorded as `unversioned` and pinned by the MD5 sum that
the data set's `checksums.md5` gives for `data.zip`, `de9fd5ca07fe2b004342aedf18357b22`, which the
data check recomputes from the stream (bucket objects dated 2026-02-11). This copy under `live/` is
the only one in the bucket (there is no `old/` folder), and the paper cites the data set by DOI with
no version, so it is the version used; the data descriptor read is version 0.6 (6 August 2019).
Session 2's spike files have no region in their folder name and the thalamic nucleus of each unit
is not in the files; where the analysis uses them, it takes them from the titles of the deposited
Matlab live scripts (`SampleScripts/*.mlx`), which the paper's Data availability statement names
as part of the deposit. Those scripts also settle a few method details the claims file leaves open;
each is cited where used.

What the data set holds: Three sessions from three rats (52, 69 and 93 min): S1 with one
thalamic (MD), six mPFC and five CA1 units and six tetrodes of multi-unit activity (MUA); S2 with
two thalamic units and the mPFC LFP only; S3, sleep after maze exploration, with two thalamic, one
mPFC and eight CA1 units, the CA1 units labelled as active or not active on the maze, and thalamic
and mPFC MUA. The paper used 32 sessions with thalamic units (63 thalamic units), six maze sessions
with sleep before and after, and four control sessions. The maze epoch, sleep before the maze,
position data and the control sessions are not in the data set.

Revision: this is the third run, after the third verification round; what changed in reply to each round is listed in the summary.

Readings: Events are used in three readings throughout: the deposited events (`dep`), the
Methods' detection rules (`methods`) and the Results' variants (`results`); see the data check.

| Claim | Status |
|---|---|
| fig1s3-spindles-not-theta | Tested |
| fig2-swr-phase-delta-spindle | Tested |
| fig2c-swr-around-kc | Tested |
| fig3a-thalamic-rate-wake-sleep | Tested |
| fig3a-bursting-in-sleep | Tested |
| fig3a-bursting-around-swrs | Tested |
| swr-shift-control | Tested |
| fig3c-thalamic-decrease-at-swrs | Tested |
| fig3s1-ca1-mpfc-at-swrs | Tested |
| glm-thalamus-anticorrelated-swrs | Tested |
| glm-re-versus-other-nuclei | Blocked: metadata insufficient |
| fig3e-thalamic-rebound-after-kc | Tested |
| kc-trough-timing | Tested |
| kc-fraction-of-thalamic-peaks | Tested |
| kc-ca1-mpfc-symmetric | Tested |
| kc-regions-compared | Tested |
| fig3f-asymmetry-index | Tested |
| spindle-mua-modulation | Tested |
| fig4-units-not-rhythmic | Tested |
| fig4b-thalamic-pairs-correlated | Tested |
| xcorr-stronger-in-spindles | Tested |
| fig5c-spindle-phase-locking | Tested |
| fig5c-delta-phase-locking | Tested |
| phase-robust-to-spike-count | Tested |
| rebound-delta-phase-trend | Tested |
| fig5d-phase-lags-consistent | Blocked: other |
| spindle-cycles-by-ripple-power | Tested |
| fig6-modulation-with-ripple-power | Tested |
| modulation-amplitude-versus-ripple | Tested |
| fig6e-quartile-synergy | Blocked: other |
| fig6f-quartile-versus-median | Tested |
| fig6g-spike-count-unchanged | Tested |
| xcorr-pairs-ripple-modulation | Blocked: other |
| thalamus-at-spindle-nested-swrs | Tested |
| mpfc-units-coupled-swrs | Tested |
| mua-coupled-swrs | Tested |
| maze-swr-kc-increase | Blocked: data missing |
| maze-control-sessions | Blocked: data missing |
| maze-ripple-power-in-spindles | Blocked: data missing |
| fig7c-run-units-in-spindles | Tested |
| maze-ca1-rates-similar | Tested |
| fig7d-run-units-peri-swr | Tested |
| fig7b-run-units-spindle-phase | Tested |
| fig8-thalamus-unchanged-by-maze | Blocked: data missing |
| fig8e-ca1-spindle-trend | Blocked: data missing |
| thalamus-not-selective-for-run-cells | Blocked: data missing |

In [ ]:
import time
import zlib
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import signal, stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

import sleepcoord as sc

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
started = time.time()

S = {}
for name in sc.SESSIONS:
    s = sc.load_session(name)
    t = s["t"]
    s["sleep_mask"] = sc.in_periods(t, s["sleep"])
    s["wake"] = sc.complement(s["sleep"], t[0], t[-1])
    s["f"] = {"delta": sc.bandpass(s["lfp"]["PFC"], sc.BANDS["delta"]),
              "spindle": sc.bandpass(s["lfp"]["PFC"], sc.BANDS["spindle"])}
    if "HC" in s["lfp"]:
        s["f"]["ripple"] = sc.bandpass(s["lfp"]["HC"], sc.BANDS["ripple"])
        s["f"]["hc_spindle"] = sc.bandpass(s["lfp"]["HC"], sc.BANDS["spindle"])
    S[s["short"]] = s
print(f"loaded {len(S)} sessions from {sc.DATASET} in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
t0 = time.time()
md5 = sc.verify_md5()
print(f"data.zip MD5 {md5} (checksums.md5 gives {sc.DATA_ZIP_MD5}): {'match' if md5 == sc.DATA_ZIP_MD5 else 'DIFFERENT'}; streamed in {time.time() - t0:.0f} s")
print("Members read:", len([m for m in sc.members() if m.filename.endswith('.mat')]), "MATLAB files of", len(sc.members()), "zip entries")

In [ ]:
rows = []
for key, s in S.items():
    for kind in ("units", "mua"):
        for u in s[kind]:
            sp = u["spikes"]
            rows.append({"session": key, "kind": kind[:-1] if kind == "units" else "mua", "region": u["region"],
                         "id": u["id"], "run": u.get("run", ""), "spikes": len(sp),
                         "first_s": round(sp[0], 1), "last_s": round(sp[-1], 1),
                         "rate_hz": round(len(sp) / (sp[-1] - sp[0]), 2), "time_sorted": u["sorted"],
                         "dup_times": int(np.sum(np.diff(sp) == 0)),
                         "amp_min": np.nanmin(u["amp"]), "amp_max": np.nanmax(u["amp"]),
                         "nan_wires": int(np.all(np.isnan(u["amp"]), axis=0).sum())})
units_df = pd.DataFrame(rows)
print(units_df.to_string(index=False))
print("\nUnits and MUA tetrodes by session and region:")
print(units_df.groupby(["session", "kind", "region"]).size().unstack(fill_value=0).to_string())
print("\nLFP (600 Hz) per session: start, end, samples, sampling interval, value ranges (units not stated):")
for key, s in S.items():
    t = s["t"]
    dt = np.diff(t)
    print(f"  {key}: {t[0]:.3f} to {t[-1]:.3f} s, {len(t)} samples, {(t[-1] - t[0]) / 60:.1f} min, "
          f"dt median {np.median(dt) * 1e3:.4f} ms (max {dt.max() * 1e3:.4f}), "
          + ", ".join(f"{k} {np.percentile(v, 0.5):.0f} to {np.percentile(v, 99.5):.0f}" for k, v in s["lfp"].items()))
    print(f"     sleep periods {len(s['sleep'])}, {sc.total_time(s['sleep']) / 60:.1f} min of sleep, "
          f"{sc.total_time(s['wake']) / 60:.1f} min outside: " + "; ".join(f"{a:.0f}-{b:.0f}" for a, b in s['sleep']))
    for k, v in s["derived"].items():
        v = np.asarray(v)
        print(f"     derived {k}: shape {v.shape}, {np.nanmin(v):.2f} to {np.nanmax(v):.2f} s")
print("\nData descriptor: three sessions of 52, 69 and 93 min; the LFP spans above are",
      [round((s['t'][-1] - s['t'][0]) / 60, 1) for s in S.values()], "min.")

The files match the descriptor: three sessions of 52.3, 69.2 and 92.7 min, LFP at 600 Hz (sample interval 1.6667 ms throughout), unit and MUA files of five columns with times in seconds, all time-sorted. Counts per session and region are printed above; with them the data set holds 5 thalamic, 7 mPFC and 13 CA1 units, against 63, about 46 and about 111 in the paper. Session 2 has only the mPFC LFP and two units in a folder with no region; they are taken as thalamic from the deposited script `PairCorrelationSleep.mlx`. The LFP files give no unit; their ranges (a few hundred) are consistent with microvolts, which is assumed where an amplitude is compared with the paper. Each session has 10 to 15 min outside the sleep periods, above the paper's 5 min minimum for the waking comparison. In S3 the mPFC unit and MUA start at 11750 s, 1450 s (24 min) before the LFP, the sleep periods and every other stream (13200 s); nothing in the descriptor says what this epoch is. The S3 MUA files hold repeated timestamps (657 and 6230), and two S3 thalamic units 17 and 8.

In [ ]:
# Do unit spikes come from the tetrode's MUA? Fraction of each unit's spike times that are also MUA times.
rows = []
for key, s in S.items():
    for u in s["units"]:
        same = [m for m in s["mua"] if m["region"] == u["region"] and m["folder"] == u["folder"]]
        if not same:
            same = [m for m in s["mua"] if m["region"] == u["region"]]
        if same:
            m = same[0]["spikes"]
            i = np.clip(np.searchsorted(m, u["spikes"]), 1, len(m) - 1)
            d = np.minimum(np.abs(m[i] - u["spikes"]), np.abs(m[i - 1] - u["spikes"]))
            rows.append({"unit": u["id"], "mua": same[0]["id"], "frac_in_mua": round(float(np.mean(d < 1e-4)), 3)})
        else:
            rows.append({"unit": u["id"], "mua": "none", "frac_in_mua": np.nan})
print(pd.DataFrame(rows).to_string(index=False))
# Duplicate recordings: pairs of units with identical spike times, across all units.
allu = [u for s in S.values() for u in s["units"]]
dups = []
for i in range(len(allu)):
    for j in range(i + 1, len(allu)):
        a, b = allu[i]["spikes"], allu[j]["spikes"]
        if abs(len(a) - len(b)) < 0.05 * max(len(a), len(b)):
            k = np.clip(np.searchsorted(b, a), 1, len(b) - 1)
            f = np.mean(np.minimum(np.abs(b[k] - a), np.abs(b[k - 1] - a)) < 1e-4)
            if f > 0.5:
                dups.append((allu[i]["id"], allu[j]["id"], f))
print("unit pairs sharing more than half their spike times:", dups if dups else "none")
# Timestamp precision: values that equal their single-precision (float32) rounding.
rows = []
for key, s in S.items():
    for u in s["units"] + s["mua"]:
        x = u["spikes"]
        d = np.diff(np.unique(x))
        rows.append({"session": key, "id": u["id"], "frac_float32": np.mean(np.float32(x).astype(float) == x), "smallest_step_ms": d[d > 0].min() * 1e3})
prec = pd.DataFrame(rows)
print(prec.groupby("session").agg(frac_float32_min=("frac_float32", "min"), frac_float32_max=("frac_float32", "max"),
                                  smallest_step_ms_max=("smallest_step_ms", "max")).round(3).to_string())
print("LFP timestamps equal to float32:", {k: float(np.mean(np.float32(s['t']).astype(float) == s['t'])) for k, s in S.items()})

Every unit's spikes are in its tetrode's MUA (fractions 0.999 to 1), and no two units share their spike times, so no recording is deposited twice; the three sessions come from three rats, so no unit is counted twice. Spike times are stored at single precision: in S3 every time equals its float32 rounding, a resolution of 2^-10 s (0.98 ms) at these values, which also produces the repeated timestamps; in S1 and S2 73 to 87 % do. The LFP and derived event times are in double precision. Histograms here use 5 ms or larger bins, so this limits only lags under about 1 ms.

In [ ]:
# Event readings. "dep": the deposited derived events. "methods": KC by M-KC (mean and sd over the
# sleep periods, k = 4), spindle troughs by M-spindle reading (a), SWRs by M-SWR with k = 3.
# "results": the same KCs, spindle troughs by reading (b), SWRs with k = 4. Session 3 deposits no
# full SWR list (only SWRs inside spindle cycles and an equal sample outside), so its SWRs come from
# M-SWR in every reading; Session 2 has no CA1 LFP and no SWRs.
EV = {}
for key, s in S.items():
    t, sm_ = s["t"], s["sleep_mask"]
    _, kidx = sc.detect_kc(t, s["f"]["delta"], sm_, sm_)
    kc_own = t[sc.refine_to_trough(s["lfp"]["PFC"], kidx)]
    sp_a, ia = sc.detect_spindle_troughs(t, s["f"]["spindle"], sm_, sm_, "a")
    sp_b, ib = sc.detect_spindle_troughs(t, s["f"]["spindle"], sm_, sm_, "b")
    sp_dep = s["sp_dep"]
    idep = np.clip(np.searchsorted(t, sp_dep), 0, len(t) - 1)
    swr3 = swr4 = np.array([])
    if "ripple" in s["f"]:
        swr3, win3 = sc.detect_swr(t, s["f"]["ripple"], sm_, sm_, 3.0)
        swr4, win4 = sc.detect_swr(t, s["f"]["ripple"], sm_, sm_, 4.0)
    swr_dep = s.get("swr_dep", swr3 if len(swr3) else np.array([]))
    EV[key] = {
        "dep": {"kc": s["kc_dep"], "sp": sp_dep, "sp_idx": idep, "swr": swr_dep},
        "methods": {"kc": kc_own, "sp": sp_a, "sp_idx": ia, "swr": swr3},
        "results": {"kc": kc_own, "sp": sp_b, "sp_idx": ib, "swr": swr4},
    }
    for r in EV[key].values():
        st, en, pb, pa = sc.spindle_cycles(t, s["f"]["spindle"], r["sp_idx"])
        ok = np.isfinite(st)
        r["cyc"] = np.c_[st[ok], en[ok]]
        r["cyc_idx"] = np.c_[pb[ok], r["sp_idx"][ok], pa[ok]]
        r["sp"] = r["sp"][ok]
READINGS = ["dep", "methods", "results"]
rows = []
for key in S:
    for r in READINGS:
        e = EV[key][r]
        rows.append({"session": key, "reading": r, "KCs": len(e["kc"]), "spindle troughs": len(e["sp"]),
                     "SWRs": len(e["swr"])})
print(pd.DataFrame(rows).to_string(index=False))

Event readings. The deposited KC counts agree with M-KC (450 against 448, 525 against 548, 923 against 918), and 91 to 92 % of the deposited KC times sit on the minimum of the raw mPFC LFP within 100 ms (printed below), so the deposited times are the LFP trough that the paper says histograms are triggered on (Results, opening section, paragraph 1); M-KC times are moved to that trough in the same way. The deposited spindle troughs are 3.7 to 6.2 times as many as M-spindle reading (a) finds, and their spindle-filtered values reach -0.7 to -1.1 sd (below), where the Methods' rule needs -3 sd, so the deposited troughs were not detected by the rule as the Methods state it. The deposited S1 SWRs (1263) are more than M-SWR gives with k = 3 (840) or 4 (626).

In [ ]:
# Clock check. Spikes and LFP events are in separate files; the down-state at KCs (a drop of mPFC
# and thalamic firing) and the CA1 burst at SWRs must line up with the event times. For each spike
# stream: PETH z (M-PETH) at the deposited events and at the same events shifted by 5, 10 and 30 s
# (labels shifted in time), and the lag of the extreme bin.
rng = np.random.default_rng(1)
rows = []
for key, s in S.items():
    streams = [(m["id"], m["region"], m["spikes"]) for m in s["mua"]] or [(u["id"], u["region"], u["spikes"]) for u in s["units"]]
    if key == "S3":
        streams += [(u["id"], u["region"], u["spikes"]) for u in s["units"] if u["region"] == "CA1"][:2]
    for sid, reg, spk in streams:
        for evname, ev, want in (("KC", EV[key]["dep"]["kc"], "min"), ("SWR", EV[key]["dep"]["swr"], "max")):
            if len(ev) == 0 or (evname == "SWR" and reg != "CA1") or (evname == "KC" and reg == "CA1"):
                continue
            row = {"session": key, "stream": sid, "event": evname}
            for shift in (0, 5, 10, 30):
                c, r, z, mu, sd = sc.peth_z(spk, ev + shift, rng, n_shifts=30)
                w = sc.window(c, -0.2, 0.2)
                zz = z[w]
                k = np.nanargmin(zz) if want == "min" else np.nanargmax(zz)
                row[f"z_shift{shift}s"] = round(float(zz[k]), 2)
                if shift == 0:
                    row["lag_ms"] = round(float(c[w][k] * 1000), 1)
            rows.append(row)
clock = pd.DataFrame(rows)
print(clock.to_string(index=False))

Spikes and LFP events share one clock in every session. Around the deposited events the CA1 MUA rises to z 147 to 179 at SWRs (S1) and the S3 CA1 units to 17 to 23, and mPFC and thalamic firing drops to z -3.4 to -34.7 at KCs, with the extreme within 45 ms of the event; with the same events shifted by 5, 10 or 30 s the extremes fall to between -4.4 and 4.0. S2 has no MUA and no CA1, and is checked on its two units at KCs (z -5.5 and -3.4 against -0.2 to -2.7 shifted).

In [ ]:
# Derived data against their description and the paper's rules.
s2, e2 = S["S2"], EV["S2"]["dep"]
dep_cyc = s2["derived"]["CycleStEnd"]
order = np.argsort(s2["sp_dep"])
own = {tuple(np.round(r, 4)) for r in e2["cyc"]}
i = np.searchsorted(e2["sp"], s2["derived"]["SP_timestamp"])
i = np.clip(i, 0, len(e2["sp"]) - 1)
d_st = np.abs(e2["cyc"][i, 0] - dep_cyc[:, 0]) * 1e3
d_en = np.abs(e2["cyc"][i, 1] - dep_cyc[:, 1]) * 1e3
print(f"S2 deposited cycle start/end against the spindle-filtered peaks around each deposited trough: "
      f"start within 2 ms for {np.mean(d_st < 2):.3f} (10 ms {np.mean(d_st < 10):.3f}), end within 2 ms for {np.mean(d_en < 2):.3f} (10 ms {np.mean(d_en < 10):.3f}); "
      f"own cycle duration median {np.median(np.diff(e2['cyc'], axis=1)) * 1e3:.1f} ms, "
      f"deposited cycle duration median {np.median(np.diff(dep_cyc, axis=1)) * 1e3:.1f} ms")

s3, e3 = S["S3"], EV["S3"]["dep"]
t3 = s3["t"]
rip2 = s3["f"]["ripple"] ** 2
cs = np.r_[0, np.cumsum(rip2)]
def cycle_power(cyc_idx):
    a, b = cyc_idx[:, 0], cyc_idx[:, 2]
    return (cs[b + 1] - cs[a]) / (b - a + 1)
hr = np.sort(s3["derived"]["SpindleTroughsHiRiPow"]); lr = np.sort(s3["derived"]["SpindleTroughsLoRiPow"])
pw = cycle_power(e3["cyc_idx"])
is_hr = np.isin(np.round(e3["sp"], 5), np.round(hr, 5))
print(f"S3 troughs: {len(hr)} high and {len(lr)} low ripple power, {len(np.intersect1d(np.round(hr, 5), np.round(lr, 5)))} in both")
print(f"S3 mean squared ripple LFP over each cycle: high {np.median(pw[is_hr]):.0f}, low {np.median(pw[~is_hr]):.0f} (medians); "
      f"split at the median of this measure agrees with the deposited labels for {np.mean((pw > np.median(pw)) == is_hr):.3f} of cycles")
for nm, ev in (("SWRwithinSpdl", s3["derived"]["SWRwithinSpdl"]), ("sampleSWRnotwithinSpdl", s3["derived"]["sampleSWRnotwithinSpdl"])):
    inside = sc.in_periods(ev, e3["cyc"])
    j = np.clip(np.searchsorted(EV["S3"]["methods"]["swr"], ev), 1, len(EV["S3"]["methods"]["swr"]) - 1)
    near = np.minimum(np.abs(EV["S3"]["methods"]["swr"][j] - ev), np.abs(EV["S3"]["methods"]["swr"][j - 1] - ev)) < 0.02
    print(f"S3 {nm}: {len(ev)} SWRs, {inside.mean():.3f} inside a deposited spindle cycle, {near.mean():.3f} within 20 ms of an M-SWR (k = 3) event")
sw3 = EV["S3"]["methods"]["swr"]
print(f"S3 M-SWR (k = 3) events inside deposited cycles: {sc.in_periods(sw3, e3['cyc']).sum()} of {len(sw3)}")
for key in ("S1", "S3"):
    print(f"{key}: deposited KCs on the raw mPFC minimum within +-100 ms: "
          f"{np.mean(np.isin(np.searchsorted(S[key]['t'], EV[key]['dep']['kc']), sc.refine_to_trough(S[key]['lfp']['PFC'], np.searchsorted(S[key]['t'], EV[key]['dep']['kc'])))):.3f}")
for key in S:
    s, e = S[key], EV[key]["dep"]
    z = (s["f"]["spindle"] - s["f"]["spindle"][s["sleep_mask"]].mean()) / s["f"]["spindle"][s["sleep_mask"]].std()
    v = z[e["sp_idx"]] if len(e["sp_idx"]) == len(z[e["sp_idx"]]) else None
    print(f"{key}: deposited spindle troughs, spindle-filtered value in sleep sd units: "
          f"max {z[np.searchsorted(s['t'], s['sp_dep'])].max():.2f}, median {np.median(z[np.searchsorted(s['t'], s['sp_dep'])]):.2f}; "
          f"M-spindle (a) needs below -3")

The derived files are consistent with their names. S2's cycle start and end times lie within 10 ms of the spindle-filtered peaks before and after each deposited trough for 97 % of cycles (within 2 ms for 53 to 65 %), median duration 100 ms. S3's 3528 high and 3528 low ripple power troughs are disjoint, the high ones have three times the ripple power of the low ones, and a median split of this attempt's ripple power agrees with the labels for 83 % of cycles. Of S3's 511 SWRs labelled as within spindles, 97 % fall inside a deposited cycle, and none of the 511 sampled outside does.

In [ ]:
# The deposited live scripts (SampleScripts/*.mlx) that this notebook cites: their titles, the lines
# that name the thalamic units, and the lines of PhaseLockingCA1UnitsPFCSpindles.mlx that set the
# phase bins, the normalisation and the Rayleigh test (M-phase reading a).
import re as _re
pats = {"PhaseLockingCA1UnitsPFCSpindles": r"^bins|binsize=|Ufasehist_SWRcoup =|UnitcorrectByprob_SWRcoup =|UnitPhhist_SWRcoup =|circ_rtest\(bins|LimBR =|LimAR =",
        "PeriEventHistogramsThreeSites": r"MD|Thalam",
        "PairCorrelationSleep": r"[Tt]halam|Pair",
        "PeriEventHistogramsUnits": r"^peri-SWR|Thalamic units",
        "AutocorrelogramUnitMUA": r"Autocorrelogram|KC_timestamp - |KC_timestamp \+ |THAL",
        "PhaseLockingSWRsAndUnits": r"Phase-Locking|MD"}
for name in [m.filename for m in sc.members() if m.filename.endswith(".mlx")]:
    lines = sc.script_text(name)
    key = name.split("/")[-1][:-4]
    print(f"{name} ({len(lines)} lines)")
    seen = set()
    for i, ln in enumerate(lines):
        if _re.search(pats[key], ln) and ln.strip() not in seen:
            seen.add(ln.strip())
            print(f"   {i + 1:4d}: {ln.strip()[:150]}")

The deposited scripts confirm what the notebook takes from them. `PeriEventHistogramsThreeSites.mlx` and `PhaseLockingSWRsAndUnits.mlx` (Session 1) name the thalamic unit and tetrode MD. `PairCorrelationSleep.mlx` (Session 2) is titled "Spindle Co-activation of Thalamic Pairs" and loads its two units as thalamic. `PhaseLockingCA1UnitsPFCSpindles.mlx` bins phases in `-pi:0.2:pi`, divides each unit's histogram by the phase distribution of the LFP in the same SWR windows, rescales it to the spike count and tests it with `circ_rtest(bins(1:end-1), hist)`, with no bin spacing, so no bin-spacing correction (M-phase reading a here follows this, with the whole sleep LFP as the reference, as the Methods say). `AutocorrelogramUnitMUA.mlx` takes spikes within 2.5 s of KCs.

In [ ]:
# What the data set holds of the paper's design. Session 3 is sleep after maze exploration: the
# first and last spike or sample of each stream, against the LFP and the sleep periods.
s3 = S["S3"]
print(f"S3 LFP {s3['t'][0]:.1f} to {s3['t'][-1]:.1f} s; sleep periods from {s3['sleep'][0, 0]:.1f} s")
for u in s3["units"] + s3["mua"]:
    sp = u["spikes"]
    print(f"  {u['id']:34s} {sp[0]:9.1f} to {sp[-1]:9.1f} s; spikes before the LFP starts: {(sp < s3['t'][0]).sum()}")
print("No file holds the maze epoch, sleep before the maze, position (LED) data, or the long control sessions.")
print("Thalamic nucleus per unit: only the S1 unit is named (MD) and only in the title of a deposited script; "
      "S2's two units sit in a folder with no region in its name (the deposited script calls them thalamic pairs).")

The data set holds no maze epoch, no sleep before the maze, no position data and none of the four long control sessions, so the claims that compare sleep before and after the maze are blocked (`data_missing`). The CA1 files labelled "InRun" hold spikes of the sleep after the maze only (from 13201 s); the label is the only information about the maze. The thalamic nucleus of each unit is not given (the S1 unit is MD by a script title), so the comparison of RE with other nuclei is blocked (`metadata_insufficient`).

## fig1s3-spindles-not-theta

> Spindle cycles detected in sleep carry spectral power concentrated near 10 to 15 Hz in the mPFC and CA1 LFP, unlike cycles detected by the same algorithm in quiet or active waking, which have theta-like profiles.

(Figure 1, figure supplement 3 legend)

In [ ]:
# fig1s3-spindles-not-theta. Morlet wavelet (7 cycles, a standard choice) power of the mPFC and CA1
# LFP in +-0.5 s around spindle troughs detected in sleep and, with the same thresholds, outside the
# sleep periods. Power at each frequency is shown relative to the mean power of the same LFP in the
# same state (a choice: raw power of an LFP falls with frequency, so its peak would sit at the lowest
# frequency whatever the event); the raw-power peak in 4 to 30 Hz is shown too.
FREQS = np.arange(2.0, 30.5, 1.0)
MAX_TROUGHS = 800   # random subset of troughs per condition (seeded), to keep the cell fast


def morlet_power(x, idx, half=300, pad=300, ncyc=7.0):
    """Mean Morlet power (frequency-domain Gaussian of sd f/7, analytic) in +-half samples."""
    idx = np.asarray([i for i in idx if i - half - pad >= 0 and i + half + pad < len(x)])
    if len(idx) == 0:
        return None
    seg = x[idx[:, None] + np.arange(-half - pad, half + pad + 1)[None, :]]
    n = seg.shape[1]
    F = np.fft.fft(seg, axis=1)
    fr = np.fft.fftfreq(n, 1 / sc.FS)
    out = np.empty((len(FREQS), n - 2 * pad))
    for k, f0 in enumerate(FREQS):
        g = 2 * np.exp(-0.5 * ((fr - f0) / (f0 / ncyc)) ** 2) * (fr > 0)
        an = np.fft.ifft(F * g, axis=1)[:, pad:-pad]
        out[k] = np.mean(np.abs(an) ** 2, axis=0)
    return out, len(idx)


rows = []
for key in ("S1", "S3"):
    s = S[key]
    t, sm_ = s["t"], s["sleep_mask"]
    wake_idx = {r: sc.detect_spindle_troughs(t, s["f"]["spindle"], sm_, ~sm_, r)[1] for r in ("a", "b")}
    conds = [(rd, "sleep", EV[key][rd]["sp_idx"]) for rd in READINGS] + \
            [(f"M-spindle ({r})", "outside sleep", wake_idx[r]) for r in ("a", "b")]
    for rd, state, idx in conds:
        if True:
            for reg, lfp in (("mPFC", s["lfp"]["PFC"]), ("CA1", s["lfp"]["HC"])):
                rng = np.random.default_rng(10)
                idx_use = idx if len(idx) <= MAX_TROUGHS else np.sort(rng.choice(idx, MAX_TROUGHS, replace=False))
                res = morlet_power(lfp, idx_use)
                if res is None:
                    continue
                pw, n = res
                # reference: mean power at random times of the same state
                pool = np.where(sm_ if state == "sleep" else ~sm_)[0]
                ref, _ = morlet_power(lfp, np.sort(rng.choice(pool[(pool > 1000) & (pool < len(t) - 1000)], 300, replace=False)))
                rel = pw.mean(1) / ref.mean(1)
                raw = pw.mean(1)
                band = (FREQS >= 4) & (FREQS <= 30)
                rows.append({"session": key, "troughs": rd,
                             "state": state, "region": reg, "n_troughs": n,
                             "peak_rel_Hz": FREQS[np.argmax(rel)], "peak_rel_ratio": round(float(rel.max()), 2),
                             "peak_raw_4_30_Hz": FREQS[band][np.argmax(raw[band])]})
fig1s3 = pd.DataFrame(rows)
print("Outside sleep, troughs are detected by the M-spindle rule with the sleep thresholds (the deposit has no waking troughs).")
print(fig1s3.to_string(index=False))

**Result: not reproduced.** No scaling of the scalogram meets both parts of the criterion. With power relative to the mean power of the same state, troughs detected in sleep carry mPFC power peaking at 10 to 11 Hz in both sessions, but troughs detected outside sleep give CA1 peaks of 13 Hz in S1 (11 to 12 Hz in sleep) and 9 to 10 Hz in S3 (9 Hz in sleep), so the waking CA1 peak is not lower. In raw power the S3 waking CA1 peak is lower (7 Hz against 8 Hz), but the S3 mPFC sleep peak falls at 4 to 8 Hz, outside 10 to 15 Hz. Outside sleep here is all time outside the deposited sleep periods in the box, which includes any REM sleep; the paper's active waking on the maze is not in the data set.

## fig2-swr-phase-delta-spindle

> CA1 SWRs are phase-locked to the mPFC delta peak (circular mean -0.04 rad) and to the trough and ascending phase of mPFC spindles (circular mean -1.72 rad).

(Results, opening section, paragraph 4; Figure 2B; Figure 2, figure supplement 1)

In [ ]:
# fig2-swr-phase-delta-spindle. Hilbert phase of the delta- and spindle-filtered mPFC LFP at SWR
# times (M-phase); per session the circular mean of the histogram normalised by the phase
# distribution of the whole filtered LFP in the sleep periods; across sessions circular mean and
# circular sd. Rayleigh test per session, readings (a) normalised histogram, (b) raw phases.
for s in S.values():
    s["phase"] = {b: np.angle(signal.hilbert(s["f"][b])) for b in ("delta", "spindle")}
    s["ref_phase"] = {b: s["phase"][b][s["sleep_mask"]][::5] for b in ("delta", "spindle")}
rows = []
for rd in READINGS:
    for key in ("S1", "S3"):
        s = S[key]
        sw = EV[key][rd]["swr"]
        for b in ("delta", "spindle"):
            ph = sc.phase_at(s["t"], s["phase"][b], sw)
            r = sc.phase_summary(ph, s["ref_phase"][b])
            rows.append({"reading": rd, "session": key, "band": b, "n_swr": len(sw), **{k: round(v, 4) for k, v in r.items() if k != "n"}})
fig2ph = pd.DataFrame(rows)
print(fig2ph.to_string(index=False))
summ = []
for (rd, b), g in fig2ph.groupby(["reading", "band"], sort=False):
    for col in ("mean_a", "mean_b"):
        m = sc.circ_mean(g[col].values)
        summ.append({"reading": rd, "band": b, "per-session mean from": "normalised hist" if col == "mean_a" else "raw phases",
                     "across-session circ mean": round(m, 3), "circ sd": round(sc.circ_std(g[col].values), 3),
                     "expected": -0.04 if b == "delta" else -1.72, "tol": 0.43 if b == "delta" else 0.93,
                     "within": sc.within(abs(np.angle(np.exp(1j * (m - (-0.04 if b == 'delta' else -1.72))))), 0, 0.43 if b == "delta" else 0.93)})
fig2sum = pd.DataFrame(summ)
print(fig2sum.to_string(index=False))

**Result: reproduced.** Across the two sessions with CA1 the circular mean delta phase at SWRs is 0.00 to 0.06 rad (normalised histograms) and the spindle phase -1.57 to -1.72 rad, all within the paper's -0.04 +- 0.43 and -1.72 +- 0.93 under every event reading. The per-session Rayleigh tests give p < 1e-19 in both sessions and bands (Rayleigh Z 45 to 205).

## fig2c-swr-around-kc

> SWR probability rises within about 250 ms before and after mPFC K-complexes (4.65 sd at 208 ms before, 3.02 sd at 267 ms after), with its minimum about 10 ms after the KC.

(Results, Thalamic rebound following the down-state, paragraph 2; Figure 2C)

In [ ]:
# fig2c-swr-around-kc. Peri-KC histogram of SWR times (M-PETH) per session (S1, S3; S2 has no CA1).
rows, curves = [], {}
for rd in READINGS:
    for sign in ("forward", "random"):
        zs = []
        for key in ("S1", "S3"):
            rng = np.random.default_rng(20)
            c, r, z, mu, sd = sc.peth_z(np.sort(EV[key][rd]["swr"]), EV[key][rd]["kc"], rng, sign=sign)
            zs.append(z)
            wb, wa = sc.window(c, -0.5, 0), sc.window(c, 0, 0.5)
            row = {"reading": rd, "shift": sign, "session": key, "n_kc": len(EV[key][rd]["kc"]), "n_swr": len(EV[key][rd]["swr"]),
                   "peak_before_z": np.nanmax(z[wb]), "t_before_ms": -c[wb][np.nanargmax(z[wb])] * 1e3,
                   "peak_after_z": np.nanmax(z[wa]), "t_after_ms": c[wa][np.nanargmax(z[wa])] * 1e3}
            for lab, (lo, hi) in (("min_t_ms_a", (-0.25, 0.25)), ("min_t_ms_b", (-0.5, 0.5))):
                w = sc.window(c, lo, hi)
                row[lab] = c[w][np.nanargmin(z[w])] * 1e3
            row["min_z_a"] = np.nanmin(z[sc.window(c, -0.25, 0.25)])
            rows.append(row)
        curves[(rd, sign)] = (c, np.nanmean(zs, axis=0))
fig2c = pd.DataFrame(rows).round(2)
print(fig2c.to_string(index=False))
EXP2C = [4.65, 208.15, 3.02, 266.85, 10.33]
TOL2C = [0.465, 90, 0.302, 100, 5.88]
summ = []
for (rd, sign), g in fig2c.groupby(["reading", "shift"], sort=False):
    c, zm = curves[(rd, sign)]
    wb, wa = sc.window(c, -0.5, 0), sc.window(c, 0, 0.5)
    for win in ("a", "b"):
        lo, hi = (-0.25, 0.25) if win == "a" else (-0.5, 0.5)
        w = sc.window(c, lo, hi)
        mins = g[f"min_t_ms_{win}"].mean()
        i_vals = [g.peak_before_z.mean(), g.t_before_ms.mean(), g.peak_after_z.mean(), g.t_after_ms.mean(), mins]
        ii_vals = [np.nanmax(zm[wb]), g.t_before_ms.mean(), np.nanmax(zm[wa]), g.t_after_ms.mean(), mins]
        for lab, vals in (("i", i_vals), ("ii", ii_vals)):
            ok = [sc.within(v, e, tl) for v, e, tl in zip(vals, EXP2C, TOL2C)]
            summ.append({"reading": rd, "shift": sign, "min window": win, "peaks": lab,
                         "before_z": round(vals[0], 2), "t_before": round(vals[1], 1), "after_z": round(vals[2], 2),
                         "t_after": round(vals[3], 1), "t_min": round(vals[4], 1), "n_within": sum(ok), "all_within": all(ok)})
fig2csum = pd.DataFrame(summ)
print(fig2csum.to_string(index=False))
print("Expected:", EXP2C, "tolerances:", TOL2C)
c, zm = curves[("dep", "forward")]
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(c, zm, "k")
ax.axhline(2, ls=":", c="grey"); ax.axhline(-2, ls=":", c="grey")
ax.set_xlabel("time from KC (s)"); ax.set_ylabel("SWR rate, z from null"); ax.set_title("peri-KC SWRs, mean of S1 and S3, deposited events")
plt.show()
print(f"plotted z ranges {np.nanmin(zm):.2f} to {np.nanmax(zm):.2f}; all points inside the axes")

**Result: partially reproduced.** SWR rate rises before and after KCs in both sessions under every reading (peaks z 2.9 to 5.1 in -0.5 to 0 s and 0 to 0.5 s), but no reading puts all five values within tolerance (at most 3 of 5): the minimum of SWR rate falls 30 to 60 ms before the KC in every session and reading (mean -37.5 to -45 ms in the +-250 ms window), against 10.33 +- 5.88 ms after it in the paper, and the peak times (112 to 180 ms before, 135 to 255 ms after) are earlier than the paper's 208 and 267 ms.

## Shared computation: peri-event histograms
Histograms of every unit and MUA stream around KCs, SWRs and spindle troughs (M-PETH), for each event reading; for the deposited events also with shifts of random sign (M-PETH reading b). Used by the claims below.

In [ ]:
# Peri-event histograms (M-PETH) of every unit and MUA stream, for each event reading. Each
# (stream, event, reading, shift sign) has its own seeded generator. Reading "results" differs from
# "methods" only for spindle troughs and SWRs (the KCs are the same).

streams = [(key, kind, u) for key, s in S.items() for kind in ("unit", "mua") for u in s["units" if kind == "unit" else "mua"]]
PETH = {}
t0 = time.time()
for key, kind, u in streams:
    for evk in ("kc", "swr", "sp"):
        for rd in READINGS:
            if evk == "kc" and rd == "results":
                continue
            ev = EV[key][rd][evk]
            if len(ev) == 0:
                continue
            for sign in (("forward", "random") if rd == "dep" and evk != "sp" else ("forward",)):
                rng = np.random.default_rng(zlib.crc32(f"{u['id']}|{evk}|{rd}|{sign}".encode()))
                PETH[(u["id"], evk, rd, sign)] = sc.peth_z(u["spikes"], ev, rng, sign=sign)
for key, kind, u in streams:
    for evk in ("kc",):
        PETH[(u["id"], evk, "results", "forward")] = PETH[(u["id"], evk, "methods", "forward")]
print(f"{len(PETH)} histograms in {time.time() - t0:.0f} s")
UNITS = pd.DataFrame([{"id": u["id"], "session": key, "kind": kind, "region": u["region"], "run": u.get("run")}
                      for key, kind, u in streams])
print(UNITS.groupby(["kind", "region"]).size().to_string())

## fig3a-thalamic-rate-wake-sleep

> Thalamic units fire less in sleep than in waking (10.10 against 7.73 spikes/s).

(Results, Thalamic cells decrease their overall firing, paragraph 2; Figure 3A)

In [ ]:
# fig3a: thalamic rates and bursts in waking and sleep. Sleep in three readings (M-sleep): (a) the
# deposited sleep periods, (b) and (c) the rule of the Methods with a 1 s or 10 s moving average
# (S2 has no CA1 LFP, so delta power alone there). Waking is reading (a) of the criterion, all time
# outside the sleep periods; the data set has no position data, so reading (b), velocity above
# 2 cm/s, cannot be run. As written the rule finds almost no sleep (below), so a variant (d) also
# fills dips shorter than 30 s in the 10 s average before the 2 min rule (a choice, standing in for
# the paper's visual confirmation of the periods).
SLEEP = {}
for key, s in S.items():
    pw = [s["f"]["delta"]] + ([s["f"]["ripple"]] if "ripple" in s["f"] else [])
    SLEEP[(key, "a")] = s["sleep"]
    for rdg, w, br in (("b", 1.0, 0.0), ("c", 10.0, 0.0), ("d", 10.0, 30.0)):
        SLEEP[(key, rdg)] = sc.detect_sleep(s["t"], pw, w, bridge=br)
rows = []
for (key, rdg), per in SLEEP.items():
    s = S[key]
    m = sc.in_periods(s["t"], per)
    rows.append({"session": key, "sleep reading": rdg, "periods": len(per), "sleep min": sc.total_time(per) / 60,
                 "outside min": (s["t"][-1] - s["t"][0] - sc.total_time(per)) / 60,
                 "overlap with deposited (fraction of this reading)": np.mean(s["sleep_mask"][m]) if m.any() else np.nan,
                 "deposited covered": np.mean(m[s["sleep_mask"]])})
print(pd.DataFrame(rows).round(3).to_string(index=False))
def spikes_in(sp, periods):
    return sp[sc.in_periods(sp, periods)]
rows = []
for (key, rdg), sleep in SLEEP.items():
    s = S[key]
    rec_start, rec_end = s["t"][0], s["t"][-1]
    wake = sc.complement(sleep, rec_start, rec_end)
    for u in s["units"]:
        if u["region"] != "THAL":
            continue
        sp = u["spikes"]
        sp = sp[(sp >= rec_start) & (sp <= rec_end)]
        bm = sc.burst_mask(sp)
        inw, ins = sc.in_periods(sp, wake), sc.in_periods(sp, sleep)
        row = {"sleep reading": rdg, "id": u["id"], "session": key, "wake_min": sc.total_time(wake) / 60,
               "rate_wake": inw.sum() / sc.total_time(wake), "rate_sleep": ins.sum() / sc.total_time(sleep),
               "burst%_wake": 100 * bm[inw].mean(), "burst%_sleep": 100 * bm[ins].mean()}
        sw = EV[key]["dep"]["swr"]
        if len(sw):
            sw = sw[sc.in_periods(sw, sleep)]
            win = np.c_[sw - 1, sw + 1]
            inu = sc.in_periods(sp, win) & ins                     # reading (a), union of windows
            row["burst%_swr_union"] = 100 * bm[inu].mean()
            g, gb = sc.SpikeGrid(sp), sc.SpikeGrid(sp[bm])
            ed = np.array([-1.0, 1.0])
            row["burst%_swr_each"] = 100 * gb.counts(sw, ed).sum() / g.counts(sw, ed).sum()   # reading (b), every window
        rows.append(row)
FIG3A = pd.DataFrame(rows)
fig3a = FIG3A[FIG3A["sleep reading"] == "a"]
print(FIG3A.round(2).to_string(index=False))
summ = []
for rdg, g in FIG3A.groupby("sleep reading"):
    n = len(g)
    row = {"sleep reading": rdg, "n": n, "min wake min": g.wake_min.min(),
           "rate wake": g.rate_wake.mean(), "sd w": g.rate_wake.std(), "rate sleep": g.rate_sleep.mean(), "sd s": g.rate_sleep.std(),
           "sleep lower": f"{(g.rate_sleep < g.rate_wake).sum()} of {n}", "Wilcoxon p rate": sc.wilcoxon(g.rate_wake, g.rate_sleep),
           "wake ok": sc.within(g.rate_wake.mean(), 10.10, 4.26), "sleep ok": sc.within(g.rate_sleep.mean(), 7.73, 3.09),
           "units >=5% burst wake %": 100 * (g["burst%_wake"] >= 5).mean(), "sleep %": 100 * (g["burst%_sleep"] >= 5).mean(),
           "burst% wake": g["burst%_wake"].mean(), "burst% sleep": g["burst%_sleep"].mean(),
           "burst higher in sleep": f"{(g['burst%_sleep'] > g['burst%_wake']).sum()} of {n}",
           "Wilcoxon p burst": sc.wilcoxon(g["burst%_wake"], g["burst%_sleep"])}
    sw = g.dropna(subset=["burst%_swr_union"])
    for col in ("burst%_swr_union", "burst%_swr_each"):
        lab = col.split("_")[-1]
        row[f"SWR {lab} %"] = sw[col].mean()
        row[f"SWR {lab} > sleep"] = f"{(sw[col] > sw['burst%_sleep']).sum()} of {len(sw)}"
        row[f"KW p {lab}"] = stats.kruskal(sw["burst%_wake"], sw["burst%_sleep"], sw[col]).pvalue
    summ.append(row)
print(pd.DataFrame(summ).round(4).T.to_string())
print("Expected: rates 10.10 +- 4.26 (waking) and 7.73 +- 3.09 (sleep) spikes/s; units with >= 5 % burst spikes 25 +- 2.5 and 77.5 +- 7.75 %; "
      "burst spikes 4.60 +- 7.97 and 18.65 +- 15.3 %; within 1 s of SWRs 25.52 +- 17.29 %. "
      f"Smallest two-sided Wilcoxon p with 5 units: {sc.min_wilcoxon_p(5)}.")

**Result: partially reproduced.** With the deposited sleep periods (reading a) all 5 thalamic units fire less in sleep (10.30 against 4.58 spikes/s), but the two-sided Wilcoxon p is 0.0625, the smallest 5 units allow, against the criterion's p < 0.05, and the sleep mean is just below its tolerance (7.73 +- 3.09, lower edge 4.64); the waking mean is within tolerance. The Methods' sleep rule as written (readings b and c, 1 s and 10 s averages) finds no sleep at all in S1 and S3 and 2 min in S2. The variant that fills dips shorter than 30 s (d) finds 19 to 44 min of sleep in S1 and S3, inside the deposited periods, but in S2, where only delta power is available, its 11 min lie outside the deposited sleep; under it the sleep mean (6.52) is within tolerance but sleep is lower in only 3 of 5 units (p = 1.0).

## fig3a-bursting-in-sleep

> Thalamic units burst more in sleep; 25% of units have at least 5% of spikes in bursts in waking against 77.5% in sleep, and burst spikes rise from 4.60% to 18.65%.

(Results, Thalamic cells decrease their overall firing, paragraph 4; Figure 3A)

The burst measures are printed in the cell of fig3a-thalamic-rate-wake-sleep above.

**Result: partially reproduced.** With the deposited sleep periods all 5 units burst more in sleep (1.54 against 15.85 % of spikes, both within tolerance), and 80 % of units have at least 5 % burst spikes in sleep (within 77.5 +- 7.75). But 0 % of units do so in waking, against 25 +- 2.5 %, and the Wilcoxon p of 0.0625 is the smallest 5 units allow, against p < 0.05. The sleep readings b and c find no sleep; variant (d) gives 3 of 5 units with more bursting in sleep.

## fig3a-bursting-around-swrs

> Thalamic bursting is higher still in the 2 s around CA1 SWRs (25.52% of spikes in bursts).

(Results, Thalamic cells decrease their overall firing, paragraph 4; Figure 3A)

The burst percentages in +-1 s of SWRs are printed in the cell of fig3a-thalamic-rate-wake-sleep above.

**Result: partially reproduced.** In the three thalamic units with SWRs the percentage of burst spikes within 1 s of SWRs (18.86 % for the union of windows, 19.94 % counting every window) is higher than in sleep in 3 of 3 units and within 25.52 +- 17.29, but the Kruskal-Wallis test across waking, sleep and SWR windows gives p = 0.177, not below 0.05.

## swr-shift-control

> Compared with SWR times shifted by more than 2 s or 5 s, the 2 s windows around SWRs hold fewer thalamic spikes and a higher percentage of burst spikes.

(Results, Thalamic cells decrease their overall firing, paragraph 4)

In [ ]:
# swr-shift-control: spikes and burst percentage in +-1 s windows at SWR times and at shifted times.
# Each SWR is shifted 20 times (independent uniform draws, forward) and the shifted windows pooled.
rng = np.random.default_rng(25)
N_DRAWS = 20
rows = []
for key in ("S1", "S3"):
    s = S[key]
    sw = EV[key]["dep"]["swr"]
    for u in s["units"]:
        if u["region"] != "THAL":
            continue
        sp = u["spikes"]
        bm = sc.burst_mask(sp)
        g, gb = sc.SpikeGrid(sp), sc.SpikeGrid(sp[bm])
        ed = np.array([-1.0, 1.0])
        row = {"id": u["id"], "SWRs": len(sw), "spikes_true": g.counts(sw, ed).mean(), "burst%_true": 100 * gb.counts(sw, ed).sum() / max(g.counts(sw, ed).sum(), 1)}
        for lab, (lo, hi) in (("2-3", (2, 3)), ("5-6", (5, 6)), ("2-10", (2, 10)), ("5-10", (5, 10))):
            sh = np.concatenate([sw + rng.uniform(lo, hi, len(sw)) for _ in range(N_DRAWS)])
            row[f"spikes_{lab}"] = g.counts(sh, ed).mean()
            row[f"burst%_{lab}"] = 100 * gb.counts(sh, ed).sum() / max(g.counts(sh, ed).sum(), 1)
        rows.append(row)
shift = pd.DataFrame(rows)
print(shift.round(2).to_string(index=False))
for lab in ("2-3", "5-6", "2-10", "5-10"):
    print(f"shift {lab} s: fewer spikes at SWRs in {(shift.spikes_true < shift[f'spikes_{lab}']).sum()} of {len(shift)}, "
          f"higher burst % in {(shift['burst%_true'] > shift[f'burst%_{lab}']).sum()} of {len(shift)}; "
          f"Wilcoxon p {sc.wilcoxon(shift.spikes_true, shift[f'spikes_{lab}']):.3f} and {sc.wilcoxon(shift['burst%_true'], shift[f'burst%_{lab}']):.3f}")
print("Smallest two-sided Wilcoxon p with", len(shift), "units:", sc.min_wilcoxon_p(len(shift)))

**Result: partially reproduced.** Against windows shifted by 20 random draws per SWR, SWR windows hold fewer spikes in 2 of 3 units and a higher burst percentage in 2 of 3 (3 of 3 for the 5 to 10 s shift), under every shift; the S3 unit cl-2 has more spikes at SWRs. With 3 units the smallest two-sided Wilcoxon p is 0.25, so the criterion's p < 0.05 cannot be reached.

### Peri-SWR classes per unit (shared)

In [ ]:
# Peri-SWR classes per unit in -100 to 100 ms (fig3c-thalamic-decrease-at-swrs): reading (a) extreme z,
# (b) mean z over the window. Decrease: below -2; pure increase: maximum above 2 with no bin below -2
# before it; rebound: a bin below -2 followed by a bin above 2.
def swr_class(rd, sign="forward"):
    rows = []
    for _, r in UNITS[(UNITS.kind == "unit") & (UNITS.session != "S2")].iterrows():
        c, rate, z, mu, sd = PETH[(r.id, "swr", rd, sign)]
        w = sc.window(c, -0.1, 0.1)
        zw = z[w]
        imax = int(np.nanargmax(zw))
        below = np.where(zw < -2)[0]
        dec_a = bool(np.nanmin(zw) < -2)
        inc_a = bool(zw[imax] > 2 and not np.any(below < imax))
        reb = bool(len(below) and np.any(zw[below[0]:] > 2))
        m = float(np.nanmean(zw))
        rows.append({"id": r.id, "session": r.session, "region": r.region, "min_z": np.nanmin(zw), "max_z": zw[imax], "mean_z": m,
                     "dec_a": dec_a, "inc_a": inc_a, "rebound": reb, "dec_b": m < -2, "inc_b": m > 2})
    return pd.DataFrame(rows)
SWRC = {(rd, sign): swr_class(rd, sign) for rd in READINGS for sign in (("forward", "random") if rd == "dep" else ("forward",))}
print(SWRC[("dep", "forward")].round(2).to_string(index=False))

## fig3c-thalamic-decrease-at-swrs

> About 55% of thalamic units reduce their firing by more than 2 sd from the null in the 200 ms around SWRs (average -4.23 sd); 14.89% show a pure increase and 8.51% a rebound after a decrease.

(Results, Thalamic cells decrease their overall firing, paragraph 5; Figure 3B-C)

In [ ]:
rows = []
for (rd, sign), d in SWRC.items():
    th = d[d.region == "THAL"]
    n = len(th)
    for rdg in ("a", "b"):
        dec = th[f"dec_{rdg}"]
        rows.append({"SWR reading": rd, "shift": sign, "window": rdg, "n": n, "decrease %": 100 * dec.mean(),
                     "mean z of decreasing": (th.min_z if rdg == "a" else th.mean_z)[dec].mean(),
                     "pure increase %": 100 * th[f"inc_{rdg}"].mean(), "rebound %": 100 * th.rebound.mean()})
print(pd.DataFrame(rows).round(2).to_string(index=False))
print("Thalamic units with SWRs: S1 one unit, S3 two (S2 has no CA1 LFP). With 3 units a percentage is 0, 33.3, 66.7 or 100; "
      "the bands 55.32 +- 5.53, 14.89 +- 1.49 and 8.51 +- 0.85 % hold none of them.")

**Result: not reproduced.** The direction fails. Of the 3 thalamic units with SWRs, one (S1) decreases below -2 sd (minimum z -3.95, reading a) and two (both S3) show a pure increase, so the pure increase class (66.7 % in reading a) is the largest, not the decrease class (33.3 %), under every SWR reading and shift; no unit shows a rebound. With 3 units none of the three expected percentages can be reached.

## fig3s1-ca1-mpfc-at-swrs

> Around SWRs, 90.18% of CA1 units increase their firing (by 19.85 sd on average), while in mPFC 30.95% of units increase and 40.48% decrease.

(Results, Thalamic cells decrease their overall firing, paragraph 5; Figure 3, figure supplement 1A)

In [ ]:
rows = []
for (rd, sign), d in SWRC.items():
    for rdg in ("a", "b"):
        ca, pf = d[d.region == "CA1"], d[d.region == "PFC"]
        inc_ca = ca[f"inc_{rdg}"]
        val = (ca.max_z if rdg == "a" else ca.mean_z)[inc_ca].mean()
        v = [100 * inc_ca.mean(), val, 100 * pf[f"inc_{rdg}"].mean(), 100 * pf[f"dec_{rdg}"].mean()]
        ok = [sc.within(x, e, tl) for x, e, tl in zip(v, [90.18, 19.85, 30.95, 40.48], [9.02, 18.65, 3.10, 4.05])]
        rows.append({"SWR reading": rd, "shift": sign, "window": rdg, "n_CA1": len(ca), "CA1 inc %": v[0], "CA1 inc mean z": v[1],
                     "CA1 dec %": 100 * ca[f"dec_{rdg}"].mean(), "n_mPFC": len(pf), "mPFC inc %": v[2], "mPFC dec %": v[3],
                     "n_within": sum(ok), "all_within": all(ok)})
fig3s1 = pd.DataFrame(rows).round(2)
print(fig3s1.to_string(index=False))
print("Expected CA1 90.18 +- 9.02 %, 19.85 +- 18.65 sd; mPFC 30.95 +- 3.10 % and 40.48 +- 4.05 %.")
d = SWRC[("dep", "forward")]
print(d[d.region.isin(["CA1", "PFC"])].groupby(["region", "session"])[["inc_a", "dec_a"]].agg(["sum", "count"]).to_string())

**Result: partially reproduced.** All 13 CA1 units increase their firing in +-100 ms of SWRs (mean maximum z 18.4 to 20.9, within 19.85 +- 18.65) and mPFC holds both increasing (2 of 7, 28.6 %, within tolerance) and decreasing units, as the criterion's direction asks. But 100 % of CA1 units is just above the tolerance (90.18 +- 9.02), and the mPFC decreasing fraction is 57.1 to 71.4 % against 40.48 +- 4.05, so under no reading are all four values within tolerance.

## glm-thalamus-anticorrelated-swrs

> In Poisson GLMs, CA1 activity varies inversely with the firing of most thalamic units (61.7%); for those units SWR rate rises by about 20.74% per 1 spike/s drop in thalamic firing.

(Results, Thalamic cells decrease their overall firing, paragraph 6)

In [ ]:
# glm-thalamus-anticorrelated-swrs: Poisson GLM log(mu) = b0 + b1 X in 250 ms bins over the sleep
# periods (Methods, Correlation quantifications, paragraph 2), Wald test at p < 0.001.
BIN = 0.25
rows = []
for rd in READINGS:
    for key in ("S1", "S3"):
        s = S[key]
        edges = np.concatenate([np.arange(a, b - BIN + 1e-9, BIN) for a, b in s["sleep"]])
        def binned(times):
            g = sc.SpikeGrid(np.sort(times), s["t"][0], s["t"][-1])
            return g.counts(edges, np.array([0.0, BIN])).ravel()
        Y = {"SWR": binned(EV[key][rd]["swr"])}
        ca1m = [m for m in s["mua"] if m["region"] == "CA1"]
        if ca1m:
            Y["CA1 MUA"] = binned(np.sort(np.concatenate([m["spikes"] for m in ca1m])))
        thal_mua = [m for m in s["mua"] if m["region"] == "THAL"]
        Xs = [(u["id"], "unit", binned(u["spikes"])) for u in s["units"] if u["region"] == "THAL"]
        Xs += [(m["id"], "MUA", binned(m["spikes"])) for m in thal_mua]
        for xid, xkind, xc in Xs:
            for yname, y in Y.items():
                X = sm.add_constant(xc / BIN)               # reading (i): spikes/s
                fit = sm.GLM(y, X, family=sm.families.Poisson()).fit()
                b1, p = fit.params[1], fit.pvalues[1]
                rows.append({"reading": rd, "session": key, "X": xid, "X kind": xkind, "response": yname, "bins": len(y),
                             "b1 per spike/s": b1, "wald p": p,
                             "change % per 1 spike/s drop (i)": 100 * (np.exp(-b1) - 1),
                             "change % per 1 spike per bin drop": 100 * (np.exp(-b1 / BIN) - 1)})
glm = pd.DataFrame(rows)
print(glm.round(5).to_string(index=False))
print("Wald p of the unit fits, in full:", {f"{r.reading} {r.X} {r.response}": f"{r['wald p']:.1e}" for _, r in glm[glm["X kind"] == "unit"].iterrows()})
print("Readings (i) X in spikes/s, change 100(exp(-b1)-1), and (ii) X in spikes per bin, change 100(exp(-0.25 b1')-1) with b1' = 4 b1, "
      "are the same number; the last column is the change per drop of one spike per 250 ms bin, a variant.")
summ = []
for (rd, resp), g in glm[glm["X kind"] == "unit"].groupby(["reading", "response"], sort=False):
    neg = (g["b1 per spike/s"] < 0) & (g["wald p"] < 0.001)
    pos = (g["b1 per spike/s"] > 0) & (g["wald p"] < 0.001)
    summ.append({"reading": rd, "response": resp, "n units": len(g), "neg sig %": 100 * neg.mean(), "pos sig": int(pos.sum()),
                 "mean change % (neg units)": g.loc[neg, "change % per 1 spike/s drop (i)"].mean()})
for rd in READINGS:
    g = glm[(glm.reading == rd) & (glm["X kind"] == "unit")]
    piv = g.pivot_table(index="X", columns="response", values=["b1 per spike/s", "wald p"])
    neg_any = ((piv["b1 per spike/s"] < 0) & (piv["wald p"] < 0.001)).any(axis=1)
    summ.append({"reading": rd, "response": "either (C)", "n units": len(piv), "neg sig %": 100 * neg_any.mean(), "pos sig": np.nan,
                 "mean change % (neg units)": np.nan})
print(pd.DataFrame(summ).round(2).to_string(index=False))
print("Expected 61.7 +- 6.17 % of units, 20.74 +- 2.07 %. CA1 MUA exists only in S1 (S3 deposits no CA1 MUA).")

**Result: not reproduced.** Of the 3 thalamic units with SWRs, 1 (S1) has a significant negative coefficient for SWR counts (33.3 %, against 61.7 +- 6.17 %), with a predicted 10.3 to 10.6 % rise in SWR rate per 1 spike/s drop (paper 20.74 +- 2.07). One S3 unit has a significant positive coefficient (p = 0.00015) and the other none (p = 0.59). CA1 MUA exists only in S1, where the unit's coefficient is negative.

## glm-re-versus-other-nuclei

> The size of the GLM effect is marginally larger in RE than in other thalamic nuclei (p=0.05), while the proportion of SWR-correlated units does not differ (p=0.159).

(Results, Thalamic cells decrease their overall firing, paragraph 6)

No file gives the thalamic nucleus of a unit (see the data check).

**Result: blocked (metadata insufficient).** The data set does not say which thalamic nucleus each unit is in.

## Shared computation: peri-KC measures

In [ ]:
# Per-unit peri-KC measures: peak z in -0.5 to 0 s and 0 to 0.5 s and their times; the time of the
# minimum z in +-0.25 s (a) and +-0.5 s (b); the asymmetry index from the raw-rate histogram.
def kc_table(rd, sign="forward"):
    rows = []
    for _, r in UNITS[UNITS.kind == "unit"].iterrows():
        c, rate, z, mu, sd = PETH[(r.id, "kc", rd, sign)]
        wb, wa = sc.window(c, -0.5, 0), sc.window(c, 0, 0.5)
        wa25, wb5 = sc.window(c, -0.25, 0.25), sc.window(c, -0.5, 0.5)
        mb, ma = np.nanmax(rate[wb]), np.nanmax(rate[wa])
        zb, za = np.nanmax(z[wb]), np.nanmax(z[wa])
        rows.append({"id": r.id, "session": r.session, "region": r.region,
                     "peak_before_z": zb, "t_before_ms": -c[wb][np.nanargmax(z[wb])] * 1e3,
                     "peak_after_z": za, "t_after_ms": c[wa][np.nanargmax(z[wa])] * 1e3,
                     "t_min_a_ms": c[wa25][np.nanargmin(z[wa25])] * 1e3, "t_min_b_ms": c[wb5][np.nanargmin(z[wb5])] * 1e3,
                     "asym_a": (ma - mb) / max(ma, mb), "asym_b": (ma - mb) / (ma + mb),
                     "asym_c": (za - zb) / max(abs(za), abs(zb)),
                     "mean_rate_before": np.nanmean(rate[wb]), "mean_rate_after": np.nanmean(rate[wa])})
    return pd.DataFrame(rows)
KC = {(rd, sign): kc_table(rd, sign) for rd in ("dep", "methods") for sign in (("forward", "random") if rd == "dep" else ("forward",))}
print(KC[("dep", "forward")].round(2).to_string(index=False))

## fig3e-thalamic-rebound-after-kc

> Thalamic units fire more after mPFC K-complexes than before (2.48 sd at about 275 ms after against 1.28 sd at about 263 ms before).

(Results, Thalamic rebound following the down-state, paragraph 1; Figure 3D-E)

In [ ]:
EXP = [2.48, 274.76, 1.28, 263.31]; TOL = [0.248, 135.33, 0.128, 145.31]
rows = []
for (rd, sign), d in KC.items():
    th = d[d.region == "THAL"]
    vals = [th.peak_after_z.mean(), th.t_after_ms.mean(), th.peak_before_z.mean(), th.t_before_ms.mean()]
    ok = [sc.within(v, e, t_) for v, e, t_ in zip(vals, EXP, TOL)]
    rows.append({"KC reading": rd, "shift": sign, "n_units": len(th), "after_z": vals[0], "t_after": vals[1],
                 "before_z": vals[2], "t_before": vals[3], "after>before": vals[0] > vals[2],
                 "units after>before": int((th.peak_after_z > th.peak_before_z).sum()), "n_within": sum(ok), "all_within": all(ok)})
fig3e = pd.DataFrame(rows).round(2)
print(fig3e.to_string(index=False))
print("Expected", EXP, "tolerance", TOL)
th = KC[("dep", "forward")]; th = th[th.region == "THAL"]
print("\nPer thalamic unit (deposited KCs, forward shifts):")
print(th[["id", "peak_before_z", "t_before_ms", "peak_after_z", "t_after_ms"]].round(2).to_string(index=False))
fig, ax = plt.subplots(1, len(th), figsize=(3 * len(th), 2.6), sharex=True)
zr = []
for a, uid in zip(ax, th.id):
    c, rate, z, mu, sd = PETH[(uid, "kc", "dep", "forward")]
    a.plot(c, z, "k"); a.axhline(2, ls=":", c="grey"); a.axvline(0, c="r", lw=0.5)
    a.set_title(uid.split("/")[0] + " " + uid.split("/")[-1], fontsize=8); a.set_xlabel("s from KC")
    zr.append((np.nanmin(z), np.nanmax(z)))
ax[0].set_ylabel("z")
plt.tight_layout(); plt.show()
print("z ranges per panel:", [(round(a, 1), round(b, 1)) for a, b in zr], "; each panel scales to its data")

**Result: partially reproduced.** On average thalamic units peak higher after KCs than before (z 2.39 against 1.42 with deposited KCs; 2.16 against 1.26 and 2.23 against 1.66 in the other readings), and the peak times are within tolerance, but the before-KC peak is outside 1.28 +- 0.128 in two readings (1.42 and 1.66) and the after-KC peak outside 2.48 +- 0.248 in two (2.16 and 2.23), so no reading has all four values within tolerance. Only 2 of the 5 units peak higher after than before.

## kc-trough-timing

> The trough of thalamic firing follows the KC (10.56 ms after), after the trough of mPFC units (-7.70 ms) and with the SWR trough (10.33 ms after).

(Results, Thalamic rebound following the down-state, paragraph 1)

In [ ]:
rows = []
for (rd, sign), d in KC.items():
    for win in ("a", "b"):
        th, pf = d[d.region == "THAL"], d[d.region == "PFC"]
        vt, vp = th[f"t_min_{win}_ms"].mean(), pf[f"t_min_{win}_ms"].mean()
        rows.append({"KC reading": rd, "shift": sign, "window": win, "thal_t_min": vt, "thal_sd": th[f"t_min_{win}_ms"].std(),
                     "n_thal": len(th), "mpfc_t_min": vp, "mpfc_sd": pf[f"t_min_{win}_ms"].std(), "n_mpfc": len(pf),
                     "thal within": sc.within(vt, 10.56, 130.75), "mpfc within": sc.within(vp, -7.70, 72.3), "thal later": vt > vp})
kctr = pd.DataFrame(rows).round(2)
print(kctr.to_string(index=False))
print("Expected thalamus 10.56 +- 130.75 ms, mPFC -7.70 +- 72.3 ms; SWR minimum (fig2c-swr-around-kc) printed above.")

**Result: reproduced.** The thalamic firing minimum falls 42 to 54 ms after KCs and the mPFC minimum 6 to 47 ms before, both within the paper's wide tolerances, with the thalamic minimum later, under every reading and window.

## kc-fraction-of-thalamic-peaks

> 38.71% of thalamic units have a significant peak (above 2 sd) after KCs against 9.68% with a significant peak before KCs; 19.35% have both.

(Results, Thalamic rebound following the down-state, paragraph 2)

In [ ]:
rows = []
for (rd, sign), d in KC.items():
    th = d[d.region == "THAL"]
    a, b = th.peak_after_z > 2, th.peak_before_z > 2
    n = len(th)
    rows.append({"KC reading": rd, "shift": sign, "n": n,
                 "after only %": 100 * (a & ~b).sum() / n, "before only %": 100 * (b & ~a).sum() / n, "both %": 100 * (a & b).sum() / n,
                 "after incl both %": 100 * a.sum() / n, "before incl both %": 100 * b.sum() / n})
print(pd.DataFrame(rows).round(1).to_string(index=False))
n = len(KC[("dep", "forward")].query("region == 'THAL'"))
print(f"With {n} units a percentage can be 0, 20, 40, 60, 80 or 100. The tolerance band of 9.68 +- 0.97 % "
      f"({9.68 - 0.97:.2f} to {9.68 + 0.97:.2f}) holds none of these, so the criterion cannot be met at this n.")

**Result: partially reproduced.** With M-KC times and disjoint groups (reading a), 40 % of the 5 units peak above 2 sd after KCs only (within 38.71 +- 3.87), 20 % have both (within 19.35 +- 1.94), and 20 % before only (outside 9.68 +- 0.97, a band that no multiple of 20 % reaches), so more units peak after than before (2 against 1). With the deposited KCs the groups tie (40 % after only, 40 % before only, none both).

## kc-ca1-mpfc-symmetric

> CA1 and mPFC units increase firing both before and after KCs, by similar amounts.

(Results, Thalamic rebound following the down-state, paragraph 2)

In [ ]:
EXP = {"CA1": (2.32, 1.49, 2.35, 1.69), "PFC": (2.33, 0.233, 2.31, 0.231)}
rows = []
for (rd, sign), d in KC.items():
    for reg in ("CA1", "PFC"):
        g = d[d.region == reg]
        eb, tb, ea, ta = EXP[reg]
        rows.append({"KC reading": rd, "shift": sign, "region": reg, "n": len(g),
                     "before_z": g.peak_before_z.mean(), "sd_b": g.peak_before_z.std(), "t_before": -g.t_before_ms.mean(),
                     "after_z": g.peak_after_z.mean(), "sd_a": g.peak_after_z.std(), "t_after": g.t_after_ms.mean(),
                     "before within": sc.within(g.peak_before_z.mean(), eb, tb), "after within": sc.within(g.peak_after_z.mean(), ea, ta)})
kcsym = pd.DataFrame(rows).round(2)
print(kcsym.to_string(index=False))
d = KC[("dep", "forward")]
print("\nBy session (deposited KCs, forward):")
print(d[d.region.isin(["CA1", "PFC"])].groupby(["region", "session"])[["peak_before_z", "peak_after_z"]].agg(["mean", "count"]).round(2).to_string())

**Result: partially reproduced.** CA1 units peak before and after KCs by similar amounts within tolerance under every reading (z 2.14 to 2.31 before, 2.05 to 2.21 after, n = 13). The mPFC values are outside the narrow tolerances (2.33 +- 0.233 and 2.31 +- 0.231) in every reading but one before the KC (2.36); after the KC mPFC is 3.35 to 3.63, with one S3 unit at 9.43.

## kc-regions-compared

> Peaks after KCs do not differ between thalamus, mPFC and CA1, while thalamic units fire significantly less before KCs than mPFC and CA1 units.

(Results, Thalamic rebound following the down-state, paragraph 3)

In [ ]:
rows = []
for (rd, sign), d in KC.items():
    g = [d[d.region == r] for r in ("THAL", "PFC", "CA1")]
    pa = stats.kruskal(*[x.peak_after_z for x in g]).pvalue
    pb = stats.kruskal(*[x.peak_before_z for x in g]).pvalue
    med_b = [x.peak_before_z.median() for x in g]
    rows.append({"KC reading": rd, "shift": sign, "KW p after": pa, "KW p before": pb,
                 "median before THAL": med_b[0], "PFC": med_b[1], "CA1": med_b[2],
                 "median after THAL": g[0].peak_after_z.median(), "PFC_a": g[1].peak_after_z.median(), "CA1_a": g[2].peak_after_z.median(),
                 "thal lowest before": med_b[0] < min(med_b[1:])})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("n per region:", {r: int((KC[('dep', 'forward')].region == r).sum()) for r in ("THAL", "PFC", "CA1")})

**Result: partially reproduced.** The peaks after KCs do not differ between regions (Kruskal-Wallis p = 0.64 to 0.74), as the paper reports. Before KCs the thalamic median is the lowest of the three regions under every reading, but the difference is not significant (p = 0.18 to 0.25, n = 5, 7 and 13).

## fig3f-asymmetry-index

> The KC asymmetry index is biased to positive values for thalamic units (0.24) and centred near zero for mPFC (-0.004) and CA1 (-0.06) units.

(Results, Thalamic rebound following the down-state, paragraph 3; Figure 3F)

In [ ]:
rows = []
for (rd, sign), d in KC.items():
    for col in ("asym_a", "asym_b", "asym_c"):
        row = {"KC reading": rd, "shift": sign, "index": col}
        for reg, e, tl in (("THAL", 0.24, 0.22), ("PFC", -0.004, 0.47), ("CA1", -0.06, 0.5)):
            v = d[d.region == reg][col]
            row[f"{reg} mean"] = v.mean(); row[f"{reg} sd"] = v.std(); row[f"{reg} ok"] = sc.within(v.mean(), e, tl)
        row["THAL Wilcoxon p vs 0"] = sc.wilcoxon(d[d.region == "THAL"][col])
        row["all ok"] = row["THAL ok"] and row["PFC ok"] and row["CA1 ok"] and row["THAL mean"] > 0
        rows.append(row)
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("Expected THAL 0.24 +- 0.22, mPFC -0.004 +- 0.47, CA1 -0.06 +- 0.5. Smallest p of a Wilcoxon test with 5 units:", sc.min_wilcoxon_p(5))
d = KC[("dep", "forward")]
print(d[["id", "region", "asym_a", "asym_b", "asym_c"]].round(2).to_string(index=False))

**Result: reproduced.** With the raw-rate index (reading a) the thalamic mean is 0.081 to 0.123, positive and within 0.24 +- 0.22, and mPFC (-0.017 to -0.006) and CA1 (-0.013 to 0.012) are within tolerance; reading (b) agrees. The thalamic index is positive in only 2 of 5 units (Wilcoxon against 0, p = 0.81 to 1.0), and the z-scored reading (c) gives thalamic means of -0.128 to 0.031, outside tolerance in two of its three rows.

## spindle-mua-modulation

> Multi-unit activity increases around mPFC spindle troughs in 84.62% of CA1 tetrodes, 53.3% of mPFC and 36% of thalamic tetrodes; individual units are less clearly modulated.

(Results, Sparse thalamic firing, paragraph 1; Figure 3G-H; Figure 3, figure supplement 1C)

In [ ]:
# spindle-mua-modulation: tetrodes whose MUA peri-trough z reaches 3 in -100..100 ms.
rows = []
for rd in ("dep", "methods", "results"):
    for _, r in UNITS[UNITS.kind == "mua"].iterrows():
        c, rate, z, mu, sd = PETH[(r.id, "sp", rd, "forward")]
        w = sc.window(c, -0.1, 0.1)
        rows.append({"reading": rd, "id": r.id, "region": r.region, "max z": np.nanmax(z[w])})
spm = pd.DataFrame(rows)
print(spm.pivot_table(index=["region", "id"], columns="reading", values="max z").round(2).to_string())
print((spm.assign(up=spm["max z"] >= 3).groupby(["reading", "region"]).up.agg(["sum", "count", "mean"])).round(3).to_string())
print("Tetrodes: CA1 3, mPFC 3, thalamus 2. With 3 tetrodes a percentage is 0, 33, 67 or 100; with 2, 0, 50 or 100; "
      "the bands 84.62 +- 8.46, 53.3 +- 5.33 and 36 +- 3.6 % hold none of them.")

**Result: not reproduced.** The order CA1 > mPFC > thalamus fails. Every CA1 and every mPFC tetrode reaches z 3 around spindle troughs under every trough reading (100 % in both regions), and the thalamic tetrodes 2 of 2 or 1 of 2; the strongest modulation is in mPFC (z 21.98 to 32.16 on one tetrode). CA1 at 100 % is outside 84.62 +- 8.46, and with 3 and 2 tetrodes the mPFC and thalamic bands cannot be reached.

## fig4-units-not-rhythmic

> Thalamic units do not fire rhythmically at spindle frequency; the autocorrelation peak is later for units (223.60 ms) than for thalamic MUA (119.30 ms), and falls in the spindle band for 63.3% of MUA curves against 17.02% of units.

(Results, Sparse thalamic firing, paragraph 2; Figure 4A; Figure 4, figure supplement 1)

In [ ]:
# fig4-units-not-rhythmic: autocorrelograms of sleep spikes to 500 ms, lag 0 removed, divided by the
# spike count. The peak is the maximum after the first minimum that follows lag 0 (Results, Sparse
# thalamic firing, paragraph 2): the first lag after which the curve rises. On a noisy curve that
# first minimum depends on smoothing, which the paper does not state, so three readings are run:
# the raw curve, and the curve smoothed with a Gaussian of 1 or 2 bins sd (smoothed over both sides
# of lag 0, so the first bins have no edge effect). The peak value is read from the raw curve.
def ac_peak(sp, bin, smooth=0):
    c, h = sc.autocorr(sp, bin, 0.6)
    h = h / len(sp)
    x = np.convolve(h, sc._gauss_kernel(smooth), mode="same") if smooth else h
    pos = (c > 0) & (c <= 0.5 + 1e-9)
    cc, hh, xx = c[pos], h[pos], x[pos]
    up = np.where(np.diff(xx) > 0)[0]
    i0 = int(up[0]) if len(up) else 0
    j = i0 + int(np.argmax(xx[i0:]))
    return cc[j] * 1e3, hh[j], cc[i0] * 1e3
# Check on a synthetic 10 Hz renewal process (intervals 100 +- 15 ms, half the spikes kept, plus Poisson spikes).
rng = np.random.default_rng(31)
syn = np.cumsum(np.clip(rng.normal(0.1, 0.015, 20000), 0.02, None))
syn = np.sort(np.r_[syn[rng.random(len(syn)) < 0.5], rng.uniform(0, syn[-1], 5000)])
print("synthetic 10 Hz process, peak lag (ms) for bins 5 and 10 ms and smoothing 0, 1, 2 bins:",
      [round(ac_peak(syn, b, k)[0]) for b in (0.005, 0.010) for k in (0, 1, 2)])
SMOOTH = (0, 1, 2)
rows = []
for key, s in S.items():
    thu = [u for u in s["units"] if u["region"] == "THAL"]
    if not thu:
        continue
    sl = lambda x: x[sc.in_periods(x, s["sleep"])]
    nmin = min(len(sl(u["spikes"])) for u in thu)
    for bin in (0.005, 0.010):
        curves = [(u["id"], "unit", sl(u["spikes"])) for u in thu]
        for m in s["mua"]:
            if m["region"] != "THAL":
                continue
            ms = sl(m["spikes"])
            curves.append((m["id"], "MUA all", ms))
            perm = rng.permutation(len(ms))
            curves += [(m["id"], "MUA sub", np.sort(ms[perm[k * nmin:(k + 1) * nmin]])) for k in range(min(len(ms) // nmin, 30))]
        for cid, kind, x in curves:
            for k in SMOOTH:
                lag, val, tmin = ac_peak(x, bin, k)
                rows.append({"session": key, "bin_ms": bin * 1e3, "smooth": k, "kind": kind, "id": cid, "lag_ms": lag, "value": val, "first_min_ms": tmin})
AC = pd.DataFrame(rows)
summ = []
for (bin, k), g in AC.groupby(["bin_ms", "smooth"]):
    u = g[g.kind == "unit"]
    msub = g[g.kind == "MUA sub"]
    for mua_kind in ("MUA all", "MUA sub"):
        m = g[g.kind == mua_kind]
        for band, (lo, hi) in (("i 71-167", (71, 167)), ("ii 50-150", (50, 150))):
            inb = lambda x: (x.lag_ms >= lo) & (x.lag_ms <= hi)
            summ.append({"bin_ms": bin, "smooth": k, "MUA lag from": mua_kind, "band": band, "n units": len(u), "n MUA curves": len(m),
                         "MUA lag": m.lag_ms.mean(), "unit lag": u.lag_ms.mean(), "MW p": sc.mannwhitney(u.lag_ms, m.lag_ms),
                         "sub MUA in band %": 100 * inb(msub).mean(), "units in band %": 100 * inb(u).mean(),
                         "MUA peak value in band": msub[inb(msub)].value.mean(), "unit peak value in band": u[inb(u)].value.mean(),
                         "MUA lag ok": sc.within(m.lag_ms.mean(), 119.30, 78), "unit lag ok": sc.within(u.lag_ms.mean(), 223.60, 113.30)})
F4 = pd.DataFrame(summ)
print(F4.round(3).to_string(index=False))
print(AC[AC.kind != "MUA sub"].round(3).to_string(index=False))
print("Subsampled MUA curves per tetrode:", AC[(AC.kind == "MUA sub") & (AC.smooth == 0)].groupby(["bin_ms", "id"]).size().to_dict(),
      "; block size = sleep spike count of the sparsest thalamic unit of the session")
# Each unit and all-spike MUA curve (5 ms bins) with its first minimum (smoothing 0, 1, 2 bins: x, triangle, square) and peak.
ids = AC[(AC.kind != "MUA sub")].drop_duplicates("id")[["session", "id"]].values
fig, axs = plt.subplots(2, 4, figsize=(13, 5), sharex=True)
yr = []
for a, (key, cid) in zip(axs.ravel(), ids):
    s = S[key]
    x = [u for u in s["units"] + s["mua"] if u["id"] == cid][0]["spikes"]
    x = x[sc.in_periods(x, s["sleep"])]
    c, h = sc.autocorr(x, 0.005, 0.5)
    h = h / len(x)
    p = c > 0
    a.plot(c[p] * 1e3, h[p], "k", lw=0.8)
    for k, mk in zip(SMOOTH, ("x", "^", "s")):
        lag, val, tmin = ac_peak(x, 0.005, k)
        a.plot(tmin, h[p][np.argmin(np.abs(c[p] * 1e3 - tmin))], mk, c="b", ms=6, mfc="none")
        a.plot(lag, val, mk, c="r", ms=6, mfc="none")
    a.axvspan(50, 150, color="0.9", zorder=0)
    a.set_title(cid, fontsize=8)
    yr.append((h[p].min(), h[p].max()))
for a in axs.ravel()[len(ids):]:
    a.axis("off")
for a in axs[-1]:
    a.set_xlabel("lag (ms)")
plt.tight_layout(); plt.show()
print("Blue: first minimum; red: peak. Each panel scales to its data; ranges:", [(round(a, 3), round(b, 3)) for a, b in yr])

**Result: not reproduced.** With the peak taken after the first minimum that follows lag 0 (the finder returns 100 ms on a synthetic 10 Hz process under every bin and smoothing), thalamic unit peaks are not significantly later than MUA peaks under any reading: mean unit lag 30 to 160 ms against MUA 32 to 194 ms, Mann-Whitney p = 0.15 to 1.0. The mean unit lag is later than the all-spike MUA lag in two readings (143 against 135 ms with 5 ms bins and smoothing 2; 84 against 55 ms with 10 ms bins and no smoothing), by far less than the paper's 223.6 against 119.3 ms. The share of subsampled MUA curves peaking in the spindle band is 5 to 52 %, against 63.3 +- 6.33, and is not higher than the units' share (0 to 60 %) in most readings. The first minimum of the unit curves often falls at 5 to 10 ms, after the burst lags, and the lags move with the smoothing (for example S2 cl-1 from 30 to 335 ms), so the measure is unstable on these sparse units.

## fig4b-thalamic-pairs-correlated

> 33% of simultaneously recorded thalamic unit pairs have a sleep cross-correlogram peak above 3 sd, at lags within a spindle cycle.

(Results, Sparse thalamic firing, paragraph 4; Figure 4B-C; Figure 4, figure supplement 2A)

In [ ]:
# Thalamic pairs (fig4b, xcorr-stronger-in-spindles, fig5d): S2 cl-1 with cl-2, S3 cl-1 with cl-2.
# Cross-correlograms of sleep spikes, 5 ms bins over +-1 s. Reading (a) z against the mean and sd of
# the same correlogram over +-1 s, peak within +-100 ms; reading (b) z against 100 correlograms with
# the second train shifted by 1.5 to 2.5 s (as M-PETH), peak within +-100 ms.
rng = np.random.default_rng(32)
def pair_z(a, b, rng):
    c, h = sc.xcorr(a, b, 0.005, 1.0)
    w = sc.window(c, -0.1, 0.1)
    za = (h - h.mean()) / h.std()
    nul = np.array([sc.xcorr(a, b, 0.005, 1.0, shift=rng.uniform(1.5, 2.5))[1] for _ in range(100)])
    zb = (h - nul.mean(0)) / nul.std(0, ddof=1)
    return np.nanmax(za[w]), c[w][np.nanargmax(za[w])] * 1e3, np.nanmax(zb[w]), c[w][np.nanargmax(zb[w])] * 1e3
rows, PAIRS = [], []
for key in ("S2", "S3"):
    s = S[key]
    th = [u for u in s["units"] if u["region"] == "THAL"]
    a, b = th[0]["spikes"], th[1]["spikes"]
    sl = lambda x: x[sc.in_periods(x, s["sleep"])]
    cyc = EV[key]["dep"]["cyc"]
    sp = EV[key]["dep"]["sp"]
    # between spindles: (a) gaps between consecutive troughs longer than 5 s; (b) spikes > 2.5 s from any trough
    gaps = np.c_[sp[:-1], sp[1:]][np.diff(sp) > 5]
    def far(x):
        j = np.clip(np.searchsorted(sp, x), 1, len(sp) - 1)
        return np.minimum(np.abs(sp[j] - x), np.abs(sp[j - 1] - x)) > 2.5
    sets = {"sleep": (sl(a), sl(b)), "in cycles": (a[sc.in_periods(a, cyc)], b[sc.in_periods(b, cyc)]),
            "between (a)": (sl(a)[sc.in_periods(sl(a), gaps)], sl(b)[sc.in_periods(sl(b), gaps)]),
            "between (b)": (sl(a)[far(sl(a))], sl(b)[far(sl(b))])}
    for nm, (x, y) in sets.items():
        za, la, zb, lb = pair_z(x, y, rng)
        rows.append({"session": key, "spikes": nm, "n_a": len(x), "n_b": len(y), "peak z (a)": za, "lag (a) ms": la, "peak z (b)": zb, "lag (b) ms": lb})
pairs = pd.DataFrame(rows)
print("Spindle cycles here are those of the deposited troughs (S2 also deposits cycle start and end times).")
print(pairs.round(2).to_string(index=False))
sl_ = pairs[pairs.spikes == "sleep"]
for r in ("a", "b"):
    print(f"reading ({r}): pairs with peak above 3 sd {int((sl_[f'peak z ({r})'] > 3).sum())} of {len(sl_)}; mean peak z {sl_[f'peak z ({r})'].mean():.2f} "
          f"(paper 33 % of 39 pairs, 2.40 +- 1.21). With 2 pairs a percentage is 0, 50 or 100.")

**Result: not reproduced.** Both thalamic pairs (S2, S3) have sleep correlogram peaks above 3 sd within 100 ms under both readings (100 % against 33 +- 3.3 %), and the mean peak z is 4.56 (reading a) or 7.24 (reading b), outside 2.40 +- 1.21. The two pairs are those the deposit's example scripts use.

## xcorr-stronger-in-spindles

> Pair cross-correlograms are stronger with spikes within spindle cycles (2.74 sd) than with sleep spikes between spindles (2.31 sd).

(Results, Sparse thalamic firing, paragraph 5; Figure 4D; Figure 4, figure supplement 2B-C)

In [ ]:
# Control for the restriction to spindle cycles (reading c, a choice): both trains kept in the
# cycles, the second unit's spikes of each cycle moved, with their time from the trough, into
# another cycle chosen by a random permutation (100 permutations). This keeps the cycle windows and
# each unit's timing within cycles, and removes only which cycles the two units share; the z of the
# in-cycle correlogram against it measures co-firing beyond the windowing.
rng = np.random.default_rng(33)
rows = []
for key in ("S2", "S3"):
    s = S[key]
    th = [u for u in s["units"] if u["region"] == "THAL"]
    cyc, tr = EV[key]["dep"]["cyc"], EV[key]["dep"]["sp"]
    def in_cyc(x):
        k = np.searchsorted(cyc[:, 0], x, side="right") - 1
        ok = (k >= 0) & (x <= cyc[np.clip(k, 0, None), 1])
        return x[ok], k[ok]
    a, _ = in_cyc(th[0]["spikes"])
    b, kb = in_cyc(th[1]["spikes"])
    c, h = sc.xcorr(a, b, 0.005, 1.0)
    w = sc.window(c, -0.1, 0.1)
    nul = []
    for _ in range(100):
        perm = rng.permutation(len(cyc))
        nul.append(sc.xcorr(a, np.sort(b - tr[kb] + tr[perm[kb]]), 0.005, 1.0)[1])
    nul = np.array(nul)
    z = (h - nul.mean(0)) / nul.std(0, ddof=1)
    zw = z[w]
    za = (h - h.mean()) / h.std()
    rows.append({"session": key, "spikes in cycles": f"{len(a)}, {len(b)}", "peak z (a)": za[w].max(), "peak z (c) cycle permutation": np.nanmax(zw),
                 "lag (c) ms": c[w][np.nanargmax(zw)] * 1e3, "null peak count": nul.mean(0)[w].max(), "observed peak count": h[w].max()})
PAIRC = pd.DataFrame(rows)
print(PAIRC.round(2).to_string(index=False))
print("Expected in-cycle mean peak z 2.74 +- 0.85; reading (c) mean", round(PAIRC["peak z (c) cycle permutation"].mean(), 2))

**Result: partially reproduced.** In both pairs the correlogram peak with spikes inside spindle cycles (z 6.90 and 6.60, reading a) is higher than between spindles (1.69 to 4.14; mean 2.98 for gaps over 5 s, within 2.31 +- 1.18). The in-cycle mean (6.75 in reading a, 20.3 in reading b) is outside 2.74 +- 0.85. Restricting both trains to the same cycles does not explain the in-cycle peak: against a null that moves the second unit's spikes into other cycles (reading c), the peaks are z 7.09 and 4.18 (mean 5.63, also outside tolerance). With 2 pairs the smallest two-sided Wilcoxon p is 0.5, so the test cannot reach p < 0.05.

## Shared computation: spike phases of thalamic units

In [ ]:
# fig5c: spike phases of sleep spikes (spikes in the sleep periods) of thalamic units (M-phase).
rng = np.random.default_rng(27)
rows = []
for key, s in S.items():
    for u in s["units"]:
        if u["region"] != "THAL":
            continue
        sp = u["spikes"][sc.in_periods(u["spikes"], s["sleep"])]
        sp = sp[(sp > s["t"][0]) & (sp < s["t"][-1])]
        sub = np.sort(rng.choice(sp, int(round(0.1 * len(sp))), replace=False))
        row = {"id": u["id"], "session": key, "spikes": len(sp)}
        for b in ("spindle", "delta"):
            ph = sc.phase_at(s["t"], s["phase"][b], sp)
            r = sc.phase_summary(ph, s["ref_phase"][b])
            r10 = sc.phase_summary(sc.phase_at(s["t"], s["phase"][b], sub), s["ref_phase"][b])
            row.update({f"{b}_mean_a": r["mean_a"], f"{b}_kappa_a": r["kappa_a"], f"{b}_p_a": r["p_a"],
                        f"{b}_mean_b": r["mean_b"], f"{b}_p_b": r["p_b"],
                        f"{b}_mean10_a": r10["mean_a"], f"{b}_mean10_b": r10["mean_b"]})
        rows.append(row)
PH = pd.DataFrame(rows)
print(PH.round(4).to_string(index=False))

## fig5c-spindle-phase-locking

> Most thalamic units (67.74%) fire preferentially on the descending phase of the mPFC spindle oscillation (circular mean 0.77 rad).

(Results, Phase-locking of thalamic units, paragraph 2; Figure 5C)

In [ ]:
n = len(PH)
for b, e_frac, t_frac, e_m, t_m in (("spindle", 67.74, 6.77, 0.77, 1.15), ("delta", 64.52, 6.45, 0.30, 1.04)):
    rows = []
    for rdg in ("a", "b"):
        sig = PH[f"{b}_p_{rdg}"] < 0.05
        pos = PH[f"{b}_mean_{rdg}"] > 0
        m = sc.circ_mean(PH[f"{b}_mean_{rdg}"].values)
        for lab, sel in (("significant", sig), ("positive mean", pos), ("both", sig & pos)):
            rows.append({"band": b, "phase reading": rdg, "fraction counts": lab, "n": n, "fraction %": 100 * sel.mean(),
                         "circ mean of unit means": m, "circ sd": sc.circ_std(PH[f"{b}_mean_{rdg}"].values),
                         "fraction ok": sc.within(100 * sel.mean(), e_frac, t_frac),
                         "mean ok": sc.within(abs(np.angle(np.exp(1j * (m - e_m)))), 0, t_m)})
    print(pd.DataFrame(rows).round(3).to_string(index=False))
for b, e_frac, t_frac in (("spindle", 67.74, 6.77), ("delta", 64.52, 6.45)):
    print(f"{b}: with {n} units a fraction is a multiple of {100 / n:.0f} %; the band {e_frac} +- {t_frac} % holds "
          f"{[round(float(v)) for v in 100 * np.arange(n + 1) / n if sc.within(v, e_frac, t_frac)] or 'none of them'}")

**Result: partially reproduced.** All 5 units prefer the descending phase (positive means), and the circular mean of preferred phases is 1.18 rad with normalised histograms (reading a of M-phase), within 0.77 +- 1.15. The fraction is outside tolerance under every count: 60 % significant (and 60 % significant with a positive mean) against 67.74 +- 6.77, a band no multiple of 20 % reaches, and 100 % with a positive mean. With raw phases (M-phase reading b) the mean is 1.93 rad, outside tolerance.

## fig5c-delta-phase-locking

> Thalamic units are phase-locked to mPFC delta near its positive peak (64.52% of units; circular mean 0.30 rad).

(Results, Phase-locking of thalamic units, paragraph 2; Figure 5C)

The delta rows are printed in the cell of fig5c-spindle-phase-locking above.

**Result: partially reproduced.** The circular mean of preferred delta phases (-0.32 rad normalised, -0.08 raw) is within 0.30 +- 1.04, near the peak, though 3 of 5 units prefer negative phases. Delta locking is present in more units than the paper reports: 4 of 5 units (80 %) are significant under both phase readings, outside 64.52 +- 6.45 %; with 5 units only 60 % would fall in that band.

## phase-robust-to-spike-count

> Preferred phases do not change when computed with 10% of the spikes.

(Results, Phase-locking of thalamic units, paragraph 2)

In [ ]:
for b in ("spindle", "delta"):
    for rdg in ("a", "b"):
        d = np.angle(np.exp(1j * (PH[f"{b}_mean10_{rdg}"] - PH[f"{b}_mean_{rdg}"])))
        print(f"{b} ({rdg}): circular difference 10% minus all spikes, median {np.median(d):.3f} rad, range {d.min():.3f} to {d.max():.3f}; "
              f"Wilcoxon on the phase values p = {sc.wilcoxon(PH[f'{b}_mean10_{rdg}'], PH[f'{b}_mean_{rdg}']):.3f}")
print("n =", len(PH), "units; smallest two-sided Wilcoxon p:", sc.min_wilcoxon_p(len(PH)))

**Result: reproduced, but the test cannot reject.** The Wilcoxon tests give p = 0.31 to 1.0 in both bands, as the criterion asks, but with 5 units the smallest two-sided p is 0.0625, so the test could not find a difference whatever the data. The preferred phases from 10 % of spikes differ from those of all spikes by a median of 0.18 to 0.33 rad (spindle) and -0.16 to -0.27 rad (delta), with single units up to 2.2 rad apart.

## rebound-delta-phase-trend

> The amplitude of the post-KC thalamic rebound tends to correlate with earlier preferred delta phase, though not significantly (r = -0.32, p=0.08); no such trend holds for firing before KCs.

(Results, Phase-locking of thalamic units, paragraph 2)

In [ ]:
d = KC[("dep", "forward")].set_index("id").drop(columns=["session", "region"])
th = PH.set_index("id").join(d)
rows = []
for meas, after, before in (("peak z", "peak_after_z", "peak_before_z"), ("mean rate", "mean_rate_after", "mean_rate_before")):
    for rdg in ("a", "b"):
        ph = th[f"delta_mean_{rdg}"].values
        ra, pa = stats.pearsonr(ph, th[after]); rb, pb = stats.pearsonr(ph, th[before])
        # circular-linear correlation (Mardia), p from chi-square with 2 df
        def cl(x, a):
            rxc, rxs, rcs = stats.pearsonr(x, np.cos(a))[0], stats.pearsonr(x, np.sin(a))[0], stats.pearsonr(np.cos(a), np.sin(a))[0]
            r = np.sqrt((rxc ** 2 + rxs ** 2 - 2 * rxc * rxs * rcs) / (1 - rcs ** 2))
            return r, stats.chi2.sf(len(x) * r ** 2, 2)
        cra, cpa = cl(th[after].values, ph); crb, cpb = cl(th[before].values, ph)
        rows.append({"measure": meas, "phase reading": rdg, "r after": ra, "p after": pa, "r before": rb, "p before": pb,
                     "circ-lin r after": cra, "p": cpa, "circ-lin r before": crb, "p ": cpb})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("n =", len(th), "thalamic units (KC peaks from deposited KCs, forward shifts). Paper: r = -0.32, p = 0.08 after; r = 0.08, p = 0.65 before.")

**Result: reproduced.** Under reading (i, a) the rebound peak z correlates negatively with the preferred delta phase but with r = -0.011 (p = 0.99), and the before-KC regression gives p = 0.17, as the criterion asks; reading (ii, a) is the same (r = -0.086, p = 0.89; before p = 0.72). Under reading (b) the sign is positive (r = 0.29 and 0.07). The negative r of the paper (-0.32) does not appear, so the outcome holds only because the criterion asks for a negative r of any size.

## fig5d-phase-lags-consistent

> Phase lags between spikes of correlated thalamic pairs within spindles are less variable than those of uncorrelated pairs (sd 2.15 against 2.50).

(Results, Phase-locking of thalamic units, paragraph 3; Figure 5D; Figure 5, figure supplement 1)

In [ ]:
# fig5d: phase lag between each spike of one unit in a spindle cycle and the closest spike of the
# other unit in the same cycle set, spindle phase difference unwrapped to -2 pi .. 2 pi (as written:
# the raw difference of two phases in -pi..pi).
rows = []
for key in ("S2", "S3"):
    s = S[key]
    th = [u for u in s["units"] if u["region"] == "THAL"]
    cyc = EV[key]["dep"]["cyc"]
    a = th[0]["spikes"][sc.in_periods(th[0]["spikes"], cyc)]
    b = th[1]["spikes"][sc.in_periods(th[1]["spikes"], cyc)]
    j = np.clip(np.searchsorted(b, a), 1, len(b) - 1)
    nb = np.where(np.abs(b[j] - a) < np.abs(b[j - 1] - a), b[j], b[j - 1])
    keep = np.abs(nb - a) < 0.1
    pa = sc.phase_at(s["t"], s["phase"]["spindle"], a[keep]); pb = sc.phase_at(s["t"], s["phase"]["spindle"], nb[keep])
    lag = pb - pa
    rows.append({"session": key, "lags": int(keep.sum()), "sd lag (rad)": np.std(lag, ddof=1), "circ sd": sc.circ_std(np.angle(np.exp(1j * lag)))})
    PAIRS.append(lag)
fig5d = pd.DataFrame(rows)
print(fig5d.round(3).to_string(index=False))
print("Pairs called correlated (sleep peak z > 3) under readings (a) and (b):",
      {r: [bool(v) for v in (pairs[pairs.spikes == 'sleep'][f'peak z ({r})'] > 3)] for r in ("a", "b")})

**Result: blocked (no uncorrelated pair).** Both thalamic pairs are correlated under both correlogram readings, so there is no uncorrelated group to compare. Their within-spindle lag sd are 2.36 and 2.33 rad (paper 2.15 for correlated pairs).

## spindle-cycles-by-ripple-power

> Spindle cycles with ripple power above the median are slightly shorter and larger, significant within most sessions but not across sessions.

(Results, Spindle activation of thalamic units modulated with CA1 SWRs, paragraph 1; Figure 6, figure supplement 1)

In [ ]:
# spindle-cycles-by-ripple-power: per cycle the duration (peak to peak), amplitude on the
# spindle-filtered mPFC LFP ((a) mean of the two peaks minus the trough, (b) larger peak minus the
# trough), and ripple power, the mean squared ripple-filtered CA1 LFP over the cycle; split at the
# session median. LFP values are taken to be in microvolts (the files do not say).
CYC = {}
rows = []
for rd in READINGS:
    for key in ("S1", "S3"):
        s = S[key]
        e = EV[key][rd]
        ci = e["cyc_idx"]
        sp_ = s["f"]["spindle"]
        cs = np.r_[0, np.cumsum(s["f"]["ripple"] ** 2)]
        pw = (cs[ci[:, 2] + 1] - cs[ci[:, 0]]) / (ci[:, 2] - ci[:, 0] + 1)
        dur = (ci[:, 2] - ci[:, 0]) / sc.FS * 1e3
        amp_a = (sp_[ci[:, 0]] + sp_[ci[:, 2]]) / 2 - sp_[ci[:, 1]]
        amp_b = np.maximum(sp_[ci[:, 0]], sp_[ci[:, 2]]) - sp_[ci[:, 1]]
        hi = pw > np.median(pw)
        if key == "S3" and rd == "dep":
            hi_dep = np.isin(np.round(e["sp"], 5), np.round(s["derived"]["SpindleTroughsHiRiPow"], 5))
        CYC[(rd, key)] = pd.DataFrame({"trough": e["sp"], "dur": dur, "amp_a": amp_a, "amp_b": amp_b, "pow": pw, "hi": hi})
        for amp in ("amp_a", "amp_b"):
            rows.append({"reading": rd, "session": key, "amplitude": amp, "cycles": len(pw),
                         "dur hi": dur[hi].mean(), "dur lo": dur[~hi].mean(), "p dur": sc.mannwhitney(dur[hi], dur[~hi]),
                         "amp hi": CYC[(rd, key)][amp][hi].mean(), "amp lo": CYC[(rd, key)][amp][~hi].mean(),
                         "p amp": sc.mannwhitney(CYC[(rd, key)][amp][hi], CYC[(rd, key)][amp][~hi])})
cyc = pd.DataFrame(rows)
print(cyc.round(4).to_string(index=False))
summ = []
for (rd, amp), g in cyc.groupby(["reading", "amplitude"], sort=False):
    v = [g["dur hi"].mean(), g["dur lo"].mean(), g["amp hi"].mean(), g["amp lo"].mean()]
    ok = [sc.within(x, e_, t_) for x, e_, t_ in zip(v, [102.17, 103.22, 229.21, 219.47], [2.01, 2.35, 102.14, 94.32])]
    summ.append({"reading": rd, "amplitude": amp, "dur hi": v[0], "dur lo": v[1], "amp hi": v[2], "amp lo": v[3],
                 "sessions hi shorter p<.05": int(((g["dur hi"] < g["dur lo"]) & (g["p dur"] < 0.05)).sum()),
                 "sessions hi larger p<.05": int(((g["amp hi"] > g["amp lo"]) & (g["p amp"] < 0.05)).sum()),
                 "across p dur": sc.mannwhitney(g["dur hi"], g["dur lo"]), "across p amp": sc.mannwhitney(g["amp hi"], g["amp lo"]),
                 "n_within": sum(ok)})
print(pd.DataFrame(summ).round(3).to_string(index=False))
print("Across sessions there are 2 sessions per group; a Mann-Whitney test on 2 against 2 values cannot give p below 0.33.")

**Result: partially reproduced.** With the deposited troughs, high ripple power cycles are shorter in both sessions (99.1 against 99.8 ms on average) and larger in both (241 against 231 uV), with the amplitude difference significant in 2 of 2 sessions and the duration in 1 of 2. The across session tests give p >= 0.05, but with 2 sessions they cannot give less than 0.33. No reading puts all four values within tolerance: the deposited troughs' durations (99.1 and 99.8 ms) are below 102.17 +- 2.01 and the other readings' amplitudes (334 to 379 uV) above it.

## fig6-modulation-with-ripple-power

> The peri-spindle modulation index of most thalamic units (64.52%) is higher in spindle cycles with ripple power above the median; 22.58% are more modulated in low ripple power cycles.

(Results, Spindle activation of thalamic units modulated with CA1 SWRs, paragraph 2; Figure 6A, D)

In [ ]:
# fig6 family (thalamic units with CA1: S1 one, S3 two). Modulation index of the raw-rate
# peri-trough histogram in -100..100 ms: (max - min) / mean (Results, Spindle activation,
# paragraph 2), from the histogram smoothed over +-1 s and then cropped (sc.modulation_index).
def mod_index(sp, troughs):
    return sc.modulation_index(lambda edges: sc.peth_counts(sp, np.sort(troughs), edges))
# Reference: the index of Poisson spikes at each unit's mean rate, at the same troughs (flat firing).
rng = np.random.default_rng(61)
rows = []
for key in ("S1", "S3"):
    s = S[key]
    for u in s["units"]:
        if u["region"] != "THAL":
            continue
        rate = len(u["spikes"]) / (s["t"][-1] - s["t"][0])
        flat = np.sort(rng.uniform(s["t"][0], s["t"][-1], rng.poisson(rate * (s["t"][-1] - s["t"][0]))))
        d = CYC[("dep", key)]
        rows.append({"id": u["id"], "rate": rate, "index, all cycles": mod_index(u["spikes"], d.trough.values),
                     "flat Poisson, all cycles": mod_index(flat, d.trough.values),
                     "flat Poisson, high cycles": mod_index(flat, d.trough.values[d.hi.values]),
                     "flat Poisson, low cycles": mod_index(flat, d.trough.values[~d.hi.values])})
print("Deposited troughs; flat-firing reference:")
print(pd.DataFrame(rows).round(3).to_string(index=False))
rows = []
for rd in READINGS:
    for key in ("S1", "S3"):
        d = CYC[(rd, key)]
        tr, pw, am = d.trough.values, d["pow"].values, d.amp_a.values
        hi_p, hi_a = d.hi.values, am > np.median(am)
        q_a, q_p = am >= np.quantile(am, 0.75), pw >= np.quantile(pw, 0.75)
        q1_a, q1_p = am <= np.quantile(am, 0.25), pw <= np.quantile(pw, 0.25)
        for u in S[key]["units"]:
            if u["region"] != "THAL":
                continue
            mi = lambda m: mod_index(u["spikes"], tr[m])
            rows.append({"reading": rd, "id": u["id"],
                         "hi pow": mi(hi_p), "lo pow": mi(~hi_p), "hi amp": mi(hi_a), "lo amp": mi(~hi_a),
                         "hi both": mi(hi_p & hi_a), "lo both": mi(~hi_p & ~hi_a),
                         "Q4 amp": mi(q_a), "Q1-3 amp": mi(~q_a), "Q1 amp": mi(q1_a),
                         "Q4 pow": mi(q_p), "Q1-3 pow": mi(~q_p), "Q1 pow": mi(q1_p),
                         # fig6e: in the top amplitude quartile split by ripple power, and the reverse;
                         # median (a) within the quartile subset, (b) over all cycles
                         "Q4amp hi pow (a)": mi(q_a & (pw > np.median(pw[q_a]))), "Q4amp lo pow (a)": mi(q_a & (pw <= np.median(pw[q_a]))),
                         "Q4amp hi pow (b)": mi(q_a & hi_p), "Q4amp lo pow (b)": mi(q_a & ~hi_p),
                         "Q4pow hi amp (a)": mi(q_p & (am > np.median(am[q_p]))), "Q4pow lo amp (a)": mi(q_p & (am <= np.median(am[q_p]))),
                         "Q4pow hi amp (b)": mi(q_p & hi_a), "Q4pow lo amp (b)": mi(q_p & ~hi_a)})
F6 = pd.DataFrame(rows)
print(F6.round(3).T.to_string())
pct = lambda a, b: 100 * (a - b) / b
# fig6-modulation-with-ripple-power
rows = []
for rd, g in F6.groupby("reading", sort=False):
    inc = g["hi pow"] > g["lo pow"]
    low_more = (~inc) & (g["lo pow"] >= g.loc[inc, "hi pow"].mean()) if inc.any() else ~inc & False
    rows.append({"reading": rd, "n": len(g), "higher in high cycles %": 100 * inc.mean(), "more modulated in low cycles %": 100 * low_more.mean(),
                 "mean high-cycle index of increasing units": g.loc[inc, "hi pow"].mean(),
                 "ok 64.52 +- 6.45": sc.within(100 * inc.mean(), 64.52, 6.45), "ok 22.58 +- 2.26": sc.within(100 * low_more.mean(), 22.58, 2.26)})
print("\nfig6-modulation-with-ripple-power:")
print(pd.DataFrame(rows).round(2).to_string(index=False))
print("Variant, S3 with its deposited high and low ripple power troughs in place of the median split:")
s3 = S["S3"]
for u in s3["units"]:
    if u["region"] == "THAL":
        print(f"  {u['id']}: index high {mod_index(u['spikes'], s3['derived']['SpindleTroughsHiRiPow']):.3f}, low {mod_index(u['spikes'], s3['derived']['SpindleTroughsLoRiPow']):.3f}")
# modulation-amplitude-versus-ripple
rows = []
for rd, g in F6.groupby("reading", sort=False):
    rows.append({"reading": rd, "amplitude split p": sc.wilcoxon(g["hi amp"], g["lo amp"]), "amp hi > lo": int((g["hi amp"] > g["lo amp"]).sum()),
                 "ripple split p": sc.wilcoxon(g["hi pow"], g["lo pow"]), "pow hi > lo": int((g["hi pow"] > g["lo pow"]).sum()),
                 "joint split p": sc.wilcoxon(g["hi both"], g["lo both"]), "joint hi > lo": int((g["hi both"] > g["lo both"]).sum()),
                 "similarity (a) high cycles p": sc.wilcoxon(g["hi amp"], g["hi pow"]),
                 "similarity (b) low p": sc.wilcoxon(g["lo amp"], g["lo pow"]), "similarity (b) high p": sc.wilcoxon(g["hi amp"], g["hi pow"]),
                 "median |hi amp - hi pow|": np.median(np.abs(g["hi amp"] - g["hi pow"]))})
print("\nmodulation-amplitude-versus-ripple (n = 3 units each):")
print(pd.DataFrame(rows).round(3).to_string(index=False))
# fig6e-quartile-synergy
rows = []
for rd, g in F6.groupby("reading", sort=False):
    for m in ("a", "b"):
        cr = pct(g[f"Q4amp hi pow ({m})"], g[f"Q4amp lo pow ({m})"])
        ca = pct(g[f"Q4pow hi amp ({m})"], g[f"Q4pow lo amp ({m})"])
        rho, prho = stats.spearmanr(cr, ca)
        rows.append({"reading": rd, "median": m, "units up with ripple in Q4 amp %": 100 * (cr > 0).mean(), "mean increase %": cr[cr > 0].mean(),
                     "ripple effect p": sc.wilcoxon(g[f"Q4amp hi pow ({m})"], g[f"Q4amp lo pow ({m})"]),
                     "units up with amp in Q4 pow %": 100 * (ca > 0).mean(), "mean increase % ": ca[ca > 0].mean(),
                     "amplitude effect p": sc.wilcoxon(g[f"Q4pow hi amp ({m})"], g[f"Q4pow lo amp ({m})"]),
                     "Spearman rho": rho, "Spearman p": prho})
print("\nfig6e-quartile-synergy (expected 61.29 +- 6.13 %, 18.15 +- 21.94 %, 32.26 +- 3.23 %, 30.00 +- 20.97 %):")
print(pd.DataFrame(rows).round(3).to_string(index=False))
# fig6f-quartile-versus-median
rows = []
for rd, g in F6.groupby("reading", sort=False):
    med_a, med_p = pct(g["hi amp"], g["lo amp"]), pct(g["hi pow"], g["lo pow"])
    for cmp_, lab in (("Q1-3", "a"), ("Q1", "b")):
        qa, qp = pct(g["Q4 amp"], g[f"{cmp_} amp"]), pct(g["Q4 pow"], g[f"{cmp_} pow"])
        fa = np.var(np.r_[g["Q4 amp"], g[f"{cmp_} amp"]], ddof=1) / np.var(np.r_[g["hi amp"], g["lo amp"]], ddof=1)
        fp = np.var(np.r_[g["Q4 pow"], g[f"{cmp_} pow"]], ddof=1) / np.var(np.r_[g["hi pow"], g["lo pow"]], ddof=1)
        dfn = 2 * len(g) - 1
        f_p = lambda f: 2 * min(stats.f.sf(f, dfn, dfn), stats.f.cdf(f, dfn, dfn))
        rows.append({"reading": rd, "against": lab, "amp quartile > median increase %": 100 * (qa > med_a).mean(),
                     "pow quartile > median increase %": 100 * (qp > med_p).mean(),
                     "mean Q4 amp index": g["Q4 amp"].mean(), "mean Q4 pow index": g["Q4 pow"].mean(),
                     "MW p Q4 amp vs Q4 pow": sc.mannwhitney(g["Q4 amp"], g["Q4 pow"]),
                     "F amp (quartile/median var)": fa, "F p amp": f_p(fa), "F pow": fp, "F p pow": f_p(fp)})
print("\nfig6f-quartile-versus-median (expected 29 +- 2.9 %, 22.6 +- 2.26 %, 1.16 +- 0.57, 0.60 +- 0.26):")
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("n = 3 units: smallest two-sided Wilcoxon p", sc.min_wilcoxon_p(3), "; percentages 0, 33.3, 66.7, 100; "
      "the F-tests compare the variance of the 6 indices (3 units x 2 groups) under the quartile split with that under the median split.")
# fig6g: spikes per cycle in low against high ripple power cycles.
rows = []
for rd in READINGS:
    for key in ("S1", "S3"):
        d = CYC[(rd, key)]
        cyc = EV[key][rd]["cyc"]
        for u in S[key]["units"]:
            if u["region"] == "THAL":
                n_in = np.searchsorted(u["spikes"], cyc[:, 1]) - np.searchsorted(u["spikes"], cyc[:, 0])
                rows.append({"reading": rd, "id": u["id"], "spikes/cycle high": n_in[d.hi.values].mean(), "spikes/cycle low": n_in[~d.hi.values].mean()})
F6G = pd.DataFrame(rows)
print(F6G.round(4).to_string(index=False))
for rd, g in F6G.groupby("reading", sort=False):
    dd = g["spikes/cycle high"] - g["spikes/cycle low"]
    print(f"{rd}: spikes per cycle higher in high cycles for {(dd > 0).sum()} of {len(g)}; median difference {np.median(dd):.4f}; "
          f"Wilcoxon p = {sc.wilcoxon(g['spikes/cycle high'], g['spikes/cycle low']):.3f}")

**Result: not reproduced.** With the modulation index taken from the histogram smoothed over +-1 s and then cropped (flat Poisson firing at the same troughs gives 0.06 to 0.17, against 0.18 to 0.75 for the units), only 1 of the 3 thalamic units with CA1 (33.3 %) has a higher index in high ripple power cycles with the deposited and the Methods' troughs, and none with the Results' troughs, against 64.52 +- 6.45 %; the S3 variant with the deposited high and low labels also gives 1 of 2. Units more modulated in low cycles are 66.7, 33.3 and 0 % across readings, against 22.58 +- 2.26. No majority increases, so not even the criterion's partial outcome holds. The direction of the two S3 units is within the noise of the index: their high and low cycle indices differ by 0.03 to 0.08 with the deposited troughs (and with S3's deposited labels), about the 0.07 to 0.10 index that flat firing gives at the same S3 troughs, so only the S1 unit's decrease (0.67 against 0.82) is clearly larger than the noise.

## modulation-amplitude-versus-ripple

> The modulation index increases significantly from low to high amplitude spindles (p=0.005) but not from low to high ripple power spindles (p=0.25); absolute index values are similar under both classifications.

(Results, Spindle activation of thalamic units modulated with CA1 SWRs, paragraph 4; Figure 6D)

The indices are printed in the cell of fig6-modulation-with-ripple-power above.

In [ ]:
# Values and directions of modulation-amplitude-versus-ripple, as the claims file's unit samples
# paragraph asks where no test can reach p < 0.05 (3 units): proposed after verification round 2
# and agreed in round 3. The paper: index up from low to high amplitude, not with ripple power, up
# with the joint split, and similar absolute values under both classifications.
rows = []
for rd, g in F6.groupby("reading", sort=False):
    d_a, d_p, d_j = g["hi amp"] - g["lo amp"], g["hi pow"] - g["lo pow"], g["hi both"] - g["lo both"]
    rows.append({"reading": rd, "amp up (units)": int((d_a > 0).sum()), "mean amp change": d_a.mean(),
                 "pow up (units)": int((d_p > 0).sum()), "mean pow change": d_p.mean(),
                 "joint up (units)": int((d_j > 0).sum()), "mean joint change": d_j.mean(),
                 "amp effect > pow effect": d_a.mean() > d_p.mean(),
                 "mean hi amp": g["hi amp"].mean(), "mean hi pow": g["hi pow"].mean()})
print(pd.DataFrame(rows).round(3).to_string(index=False))


**Result: partially reproduced.** With 3 units the smallest two-sided Wilcoxon p is 0.25, so the claim's contrast of a significant amplitude effect with a non-significant ripple effect cannot be tested, and the claim is rated on its values and directions, as the claims file's unit samples paragraph asks and as for kc-regions-compared (proposed after verification round 2, agreed in round 3). The index is higher for high amplitude cycles in 3, 2 and 2 of 3 units across readings (p = 0.25 to 0.75; mean change +0.09 to +0.14), for high ripple power cycles in 1, 1 and 0 of 3 (p = 0.25 to 0.75; mean change -0.03 to -0.13), and for the joint split in 3, 3 and 0 of 3 (p = 0.25; mean change +0.10, +0.06 and -0.12). The amplitude effect goes the paper's way and is larger than the ripple effect in every reading, and the ripple effect is absent. The similarity tests (p = 0.25 to 1.0) carry no information at n = 3, where p >= 0.25 is certain; the only evidence that the absolute values are similar is the median absolute difference of high-cycle indices, 0.017 to 0.063, and the mean high-cycle index is higher under the amplitude split in every reading. The two S3 units' amplitude changes with the deposited troughs are close to the index that flat firing gives at the same troughs (cell above the claim sections). Spindle amplitude is reading (a) of spindle-cycles-by-ripple-power only; reading (b) moves with it in every session but is not used here. No effect reaches significance, so the claim holds in part.

## fig6e-quartile-synergy

> In the highest amplitude spindles ripple power has no significant effect on modulation (p=0.94), while in the highest ripple power spindles amplitude does (p=0.04); 61.29% of units still increase modulation with ripple power in high amplitude spindles.

(Results, Spindle activation of thalamic units modulated with CA1 SWRs, paragraph 4; Figure 6E; Figure 6, figure supplement 2)

The quartile indices are printed in the cell of fig6-modulation-with-ripple-power above.

In [ ]:
# Values and directions of fig6e-quartile-synergy (change of outcome proposed after verification
# round 2, not agreed in round 3). The 61.29 +- 6.13 band holds 66.7 % and the 32.26 +- 3.23 band holds 33.3 %, so with
# 3 units both percentages can fall within tolerance.
rows = []
for rd, g in F6.groupby("reading", sort=False):
    for m in ("a", "b"):
        cr = pct(g[f"Q4amp hi pow ({m})"], g[f"Q4amp lo pow ({m})"])
        ca = pct(g[f"Q4pow hi amp ({m})"], g[f"Q4pow lo amp ({m})"])
        v = [100 * (cr > 0).mean(), cr[cr > 0].mean(), 100 * (ca > 0).mean(), ca[ca > 0].mean()]
        ok = [sc.within(x, e, t) for x, (e, t) in zip(v, ((61.29, 6.13), (18.15, 21.94), (32.26, 3.23), (30.0, 20.97)))]
        rows.append({"reading": rd, "median": m, "ripple up %": v[0], "ok": ok[0], "mean incr": v[1], "ok ": ok[1],
                     "amp up %": v[2], "ok  ": ok[2], "mean incr ": v[3], "ok   ": ok[3], "values ok": sum(ok),
                     "more units up with ripple than amp": v[0] > v[2], "Spearman rho": stats.spearmanr(cr, ca)[0]})
print(pd.DataFrame(rows).round(2).to_string(index=False))
print("With 3 units Spearman rho can only be -1, -0.5, 0.5 or 1.")


**Result: blocked (too few units for the claim's test); a change to not reproduced was proposed and not agreed.** With 3 units no Wilcoxon test can reach p < 0.05. The quantities are computed: in the top amplitude quartile, 33.3 to 66.7 % of units increase their index with ripple power (mean increase 24.5 to 65.1 %), and in the top ripple power quartile 66.7 to 100 % increase with amplitude (15.9 to 65.5 %); the ripple effect gives p = 0.5 to 1.0, the amplitude effect p = 0.25 to 1.0, and the Spearman correlation of the two changes is negative in 5 of 6 readings (rho -1.0 to 0.5). Proposed change, after verification round 2, rating the claim on its values and directions as for the other claims at this n: the earlier statement that the 61.29 and 32.26 % bands hold none of the possible percentages was wrong, as 66.7 % and 33.3 % fall within them. Under two readings three of the four values are within tolerance, but the share of units that increase with amplitude (66.7 to 100 %) is outside 32.26 +- 3.23 in every reading, at least as many units increase with amplitude as with ripple power in every reading, where the paper has 61 % against 32 %, and the correlation the paper reports as positive (0.51, p = 0.003) is negative in 5 of 6 readings. With 3 units rho can take only four values, so the sign is weak evidence, but no reading meets the claim's pattern. Verification round 3 did not agree and would rate the claim partially reproduced: the ripple percentage (66.7 % in 5 of 6 readings) and its mean increase (4 of 6) are within tolerance, three of four values are within tolerance in two readings, and the mean amplitude change exceeds the ripple change in the deposited and Methods readings; it reads the comparison of unit counts as the failed amplitude percentage counted twice. The outcome stays blocked, with the proposal and the verifier's view recorded. Spindle amplitude is reading (a) only.

## fig6f-quartile-versus-median

> Modulation values in the upper quartile are higher for spindle amplitude (1.16) than for ripple power (0.60), and the quartile split widens the spread of the index for amplitude but not for ripple power.

(Results, Spindle activation of thalamic units modulated with CA1 SWRs, paragraph 5; Figure 6F)

The quartile indices are printed in the cell of fig6-modulation-with-ripple-power above.

In [ ]:
# Values and directions of fig6f-quartile-versus-median (proposed after verification round 2,
# agreed in round 3): amplitude quartile index above the ripple power one, the four values,
# and whether the quartile split widens the spread more for amplitude (F ratio) than for ripple power.
rows = []
for rd, g in F6.groupby("reading", sort=False):
    med_a, med_p = pct(g["hi amp"], g["lo amp"]), pct(g["hi pow"], g["lo pow"])
    for cmp_, lab in (("Q1-3", "a"), ("Q1", "b")):
        qa, qp = pct(g["Q4 amp"], g[f"{cmp_} amp"]), pct(g["Q4 pow"], g[f"{cmp_} pow"])
        v = [100 * (qa > med_a).mean(), 100 * (qp > med_p).mean(), g["Q4 amp"].mean(), g["Q4 pow"].mean()]
        ok = [sc.within(x, e, t) for x, (e, t) in zip(v, ((29, 2.9), (22.6, 2.26), (1.16, 0.57), (0.60, 0.26)))]
        fa = np.var(np.r_[g["Q4 amp"], g[f"{cmp_} amp"]], ddof=1) / np.var(np.r_[g["hi amp"], g["lo amp"]], ddof=1)
        fp = np.var(np.r_[g["Q4 pow"], g[f"{cmp_} pow"]], ddof=1) / np.var(np.r_[g["hi pow"], g["lo pow"]], ddof=1)
        rows.append({"reading": rd, "against": lab, "Q4 amp > Q4 pow (units)": int((g["Q4 amp"] > g["Q4 pow"]).sum()),
                     "mean Q4 amp > mean Q4 pow": v[2] > v[3], "values ok": ok, "F amp": fa, "F pow": fp, "F amp > F pow": fa > fp,
                     "F amp > 1": fa > 1})
print(pd.DataFrame(rows).round(3).to_string(index=False))


**Result: partially reproduced.** With 3 units the 29 and 22.6 % bands hold none of the possible percentages, and the Mann-Whitney test of 3 against 3 values cannot give p below 0.1, so the claim is rated on its values and directions, as for the other claims at this n (proposed after verification round 2, agreed in round 3). The amplitude quartile index is above the ripple power one in every reading (in 2 or 3 of 3 units); the mean upper-quartile index is 0.51 to 0.93 for amplitude, within 1.16 +- 0.57 in the Methods and Results readings and outside it with the deposited troughs, and 0.46 to 0.55 for ripple power, within 0.60 +- 0.26 in every reading (Mann-Whitney p = 0.4 to 0.7). The spread ratio (quartile against median split) is larger for amplitude than for ripple power in 5 of 6 readings (1.18 to 2.37 against 0.79 to 1.39); the ripple power ratio is also above 1 in 4 of 6, so the quartile split widens the spread for both, and more for amplitude. The F-tests give p = 0.37 to 0.86 (amplitude) and 0.73 to 0.98 (ripple power), and the quartile split raises the index more than the median split for 33.3 to 100 % of units for amplitude and 33.3 to 66.7 % for ripple power, outside both bands. Spindle amplitude is reading (a) only. The two percentages are outside tolerance and no test is significant, so the claim holds in part.

## fig6g-spike-count-unchanged

> The number of spikes of thalamic units does not change significantly between low and high ripple power spindles, nor between SWRs coupled and uncoupled to spindles.

(Results, Spindle activation of thalamic units modulated with CA1 SWRs, paragraph 6; Figure 6G)

Spikes per cycle are printed in the cell of fig6-modulation-with-ripple-power, and rates at nested and other SWRs in the cell of thalamus-at-spindle-nested-swrs below.

**Result: reproduced, but the tests cannot reject.** Neither Wilcoxon test is significant: spikes per cycle in high against low ripple power cycles p = 0.25 to 1.0 (median difference 0.007 to 0.033 spikes, higher in high cycles for 3, 2 and 2 of 3 units), and the rate within 50 ms of nested against other SWRs p = 0.5 to 1.0 (median difference 0.33 to 1.47 spikes/s). With 3 units the smallest two-sided p is 0.25, so the tests could not find a difference whatever the data.

## xcorr-pairs-ripple-modulation

> In spindle cycles with high ripple power, the modulation index of pair cross-correlograms rises by 26.64% in 48.72% of thalamic pairs and falls by 20.90% in the rest.

(Results, Spindle activation of thalamic units modulated with CA1 SWRs, paragraph 7)

In [ ]:
# xcorr-pairs-ripple-modulation: the S3 pair (the only pair with CA1). Cross-correlogram of the two
# units' spikes inside high and inside low ripple power cycles, raw counts, modulation index over
# -100..100 ms of lag as in fig6-modulation-with-ripple-power (15 ms bins, smoothed over +-1 s,
# cropped). The z-scored correlograms of fig4b readings (a) and (b) have a mean near 0, so the index
# is taken on counts.
s3 = S["S3"]
th3 = [u for u in s3["units"] if u["region"] == "THAL"]
rows = []
for rd in READINGS:
    d = CYC[(rd, "S3")]
    cyc = EV["S3"][rd]["cyc"]
    hi = d.hi.values
    if rd == "dep":
        hi_lab = np.isin(np.round(d.trough.values, 5), np.round(s3["derived"]["SpindleTroughsHiRiPow"], 5))
    for split, h_ in (("median", hi),) + ((("deposited labels", hi_lab),) if rd == "dep" else ()):
        row = {"reading": rd, "split": split}
        for lab, m in (("high", h_), ("low", ~h_)):
            a = th3[0]["spikes"][sc.in_periods(th3[0]["spikes"], cyc[m])]
            b = th3[1]["spikes"][sc.in_periods(th3[1]["spikes"], cyc[m])]
            cnt = lambda edges: np.diff(np.searchsorted(b, (a[:, None] + edges[None, :]).ravel()).reshape(len(a), -1), axis=1).sum(0)
            row[f"index {lab}"] = sc.modulation_index(cnt)
            row[f"spikes {lab}"] = f"{len(a)}, {len(b)}"
        row["change %"] = 100 * (row["index high"] - row["index low"]) / row["index low"]
        rows.append(row)
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("Expected: 48.72 +- 4.87 % of pairs increase, by 26.64 +- 2.66 %; the rest decrease by 20.90 +- 2.09 %. With one pair the percentage is 0 or 100.")

**Result: blocked (too few pairs).** Only the S3 pair has CA1, so the percentage of pairs can be 0 or 100 and the 48.72 +- 4.87 % cannot be tested. For that pair the correlogram index falls by 8.9 % (median split) or 15.2 % (deposited labels) in high ripple power cycles with the deposited troughs, and rises by 200 % and 63 % with the Methods' and the Results' troughs, which hold far fewer spikes (147 to 712 per unit).

## Shared computation: SWRs coupled and not coupled to spindle cycles

In [ ]:
# Coupled (nested) SWRs: SWRs inside a spindle cycle; uncoupled: an equal random sample of SWRs
# outside cycles (seeded). For S3 the deposited sets are used in reading "dep".
NEST = {}
for rd in READINGS:
    for key in ("S1", "S3"):
        rng = np.random.default_rng(37)
        sw = EV[key][rd]["swr"]
        if key == "S3" and rd == "dep":
            cpl, unc = np.sort(S[key]["derived"]["SWRwithinSpdl"]), np.sort(S[key]["derived"]["sampleSWRnotwithinSpdl"])
        else:
            inside = sc.in_periods(sw, EV[key][rd]["cyc"])
            cpl = sw[inside]
            unc = np.sort(rng.choice(sw[~inside], min(len(cpl), (~inside).sum()), replace=False))
        NEST[(rd, key)] = (cpl, unc)
print({f"{k[0]} {k[1]}": (len(v[0]), len(v[1])) for k, v in NEST.items()}, "(coupled, uncoupled sample)")

def coupled_z(sp, cpl, unc, rng, null="shift"):
    """2 ms bins over +-400 ms, no smoothing; both histograms z-scored to one null from the
    union of coupled and uncoupled SWRs, (a) shifted by 1.5-2.5 s as M-PETH, (b) placed at uniform
    random times in sleep. Value: mean z in -40..40 ms (Figure 6, figure supplement 3A legend)."""
    edges = (np.arange(-200, 202) - 0.5) * 0.002
    c = (edges[:-1] + edges[1:]) / 2
    allsw = np.sort(np.r_[cpl, unc])
    nul = []
    for _ in range(100):
        if null == "shift":
            ev = allsw + rng.uniform(1.5, 2.5)
        else:
            ev = np.sort(rng.choice(sleep_pool, len(allsw)))
        nul.append(sc.peth_counts(sp, ev, edges) / 0.002)
    nul = np.array(nul)
    mu, sd = nul.mean(0), nul.std(0, ddof=1)
    w = (c >= -0.04) & (c <= 0.04)
    z = lambda ev: np.nanmean(((sc.peth_counts(sp, ev, edges) / 0.002 - mu) / np.where(sd > 0, sd, np.nan))[w])
    return z(cpl), z(unc)

## thalamus-at-spindle-nested-swrs

> Thalamic firing near SWRs is low and does not differ between spindle-nested and other SWRs in a 100 ms window (3.83 against 3.99 spikes/s), with a slight decrease for nested SWRs in windows of 1 s or more.

(Results, Spindle activation of thalamic units modulated with CA1 SWRs, paragraph 8)

In [ ]:
rows = []
for rd in READINGS:
    for key in ("S1", "S3"):
        cpl, unc = NEST[(rd, key)]
        for u in S[key]["units"]:
            if u["region"] != "THAL":
                continue
            g = sc.SpikeGrid(u["spikes"])
            r = lambda ev, h: g.counts(ev, np.array([-h, h])).mean() / (2 * h)
            rows.append({"reading": rd, "id": u["id"], "rate100 nested": r(cpl, 0.05), "rate100 other": r(unc, 0.05),
                         "rate1s nested": r(cpl, 0.5), "rate1s other": r(unc, 0.5)})
TN = pd.DataFrame(rows)
print(TN.round(3).to_string(index=False))
for rd, g in TN.groupby("reading", sort=False):
    print(f"{rd}: 100 ms {g['rate100 nested'].mean():.2f} against {g['rate100 other'].mean():.2f} spikes/s (paper 3.83 and 3.99), "
          f"1 s {g['rate1s nested'].mean():.2f} against {g['rate1s other'].mean():.2f} (paper 3.73 and 3.94); "
          f"nested lower at 1 s in {(g['rate1s nested'] < g['rate1s other']).sum()} of {len(g)}")
print("n = 3 units; smallest two-sided Wilcoxon p", sc.min_wilcoxon_p(3))
for rd, g in TN.groupby("reading", sort=False):
    dd = g["rate100 nested"] - g["rate100 other"]
    print(f"fig6g-spike-count-unchanged, {rd}: rate in +-50 ms of nested minus other SWRs, median {np.median(dd):.3f} spikes/s, "
          f"higher at nested in {(dd > 0).sum()} of {len(g)}, Wilcoxon p = {sc.wilcoxon(g['rate100 nested'], g['rate100 other']):.3f}; "
          f"1 s window: nested lower in {(g['rate1s nested'] < g['rate1s other']).sum()} of {len(g)}, Wilcoxon p = {sc.wilcoxon(g['rate1s nested'], g['rate1s other']):.3f}")

**Result: partially reproduced.** All four mean rates are within the paper's wide tolerances (100 ms: 3.50 to 3.59 nested, 2.10 to 3.90 other; 1 s: 2.85 to 2.93 nested, 2.84 to 3.02 other). The slight decrease for nested SWRs in the 1 s window appears in the means with the deposited events (2.85 against 3.02) and the Results' events (2.93 against 3.01), not with the Methods' events, and in 1 or 2 of 3 units. With 3 units the smallest two-sided Wilcoxon p is 0.25, so the decrease cannot reach p < 0.05.

## mpfc-units-coupled-swrs

> mPFC units as a population do not fire differently at SWRs coupled or uncoupled to spindles (z 0.17 against 0.24, p=0.46).

(Results, Spindle activation of thalamic units modulated with CA1 SWRs, paragraph 9; Figure 6, figure supplement 3A)

In [ ]:
rows = []
for rd in READINGS:
    for key in ("S1", "S3"):
        s = S[key]
        sleep_pool = s["t"][s["sleep_mask"]][::60]
        cpl, unc = NEST[(rd, key)]
        for kind in ("units", "mua"):
            for u in s[kind]:
                for null in ("shift", "random"):
                    rng = np.random.default_rng(zlib.crc32(f"{u['id']}|{rd}|{null}".encode()))
                    zc, zu = coupled_z(u["spikes"], cpl, unc, rng, null)
                    rows.append({"reading": rd, "null": null, "kind": kind, "id": u["id"], "region": u["region"], "z coupled": zc, "z uncoupled": zu})
CPL = pd.DataFrame(rows)
summ = []
for (rd, null), g in CPL[(CPL.kind == "units") & (CPL.region == "PFC")].groupby(["reading", "null"], sort=False):
    summ.append({"reading": rd, "null": null, "n": len(g), "z coupled": g["z coupled"].mean(), "sd c": g["z coupled"].std(),
                 "z uncoupled": g["z uncoupled"].mean(), "sd u": g["z uncoupled"].std(),
                 "Wilcoxon p": sc.wilcoxon(g["z coupled"], g["z uncoupled"]),
                 "median diff": np.median(g["z coupled"] - g["z uncoupled"]),
                 "c ok": sc.within(g["z coupled"].mean(), 0.17, 1.26), "u ok": sc.within(g["z uncoupled"].mean(), 0.24, 0.86)})
print(pd.DataFrame(summ).round(3).to_string(index=False))
print(CPL[(CPL.kind == "units") & (CPL.region == "PFC") & (CPL.reading == "dep") & (CPL.null == "shift")].round(2).to_string(index=False))

**Result: reproduced.** Over 7 mPFC units the mean z in -40 to 40 ms is -0.08 to 0.15 at coupled and -0.18 to 0.11 at uncoupled SWRs, within tolerance, and the Wilcoxon test finds no difference under any reading (p = 0.30 to 0.81; median difference 0.06 to 0.28).

## mua-coupled-swrs

> mPFC and CA1 multi-unit activity is higher at SWRs coupled to spindles than at uncoupled SWRs, with no difference in thalamic tetrodes.

(Results, Spindle activation of thalamic units modulated with CA1 SWRs, paragraph 9; Figure 6, figure supplement 3B)

In [ ]:
summ = []
for (rd, null, reg), g in CPL[CPL.kind == "mua"].groupby(["reading", "null", "region"], sort=False):
    summ.append({"reading": rd, "null": null, "region": reg, "tetrodes": len(g), "z coupled": g["z coupled"].mean(), "z uncoupled": g["z uncoupled"].mean(),
                 "coupled higher": int((g["z coupled"] > g["z uncoupled"]).sum()), "Wilcoxon p": sc.wilcoxon(g["z coupled"], g["z uncoupled"])})
print(pd.DataFrame(summ).round(3).to_string(index=False))
print("Tetrodes with SWRs: CA1 3 (S1), mPFC 3, thalamus 2; smallest two-sided Wilcoxon p with 3 tetrodes:", sc.min_wilcoxon_p(3))

**Result: partially reproduced.** CA1 MUA is higher at coupled SWRs on 3 of 3 tetrodes under every reading and null, mPFC MUA on 2 or 3 of 3, and the 2 thalamic tetrodes are mixed, as the claim states. With 3 tetrodes the smallest two-sided Wilcoxon p is 0.25, so the increases cannot reach p < 0.05.

## maze-swr-kc-increase

> After maze exploration, SWR amplitude and rate and KC amplitude increase in sleep, while spindle number and amplitude do not change.

(Results, Multi-region correlations after spatial exploration, paragraph 1)

The data set holds no sleep before the maze (see the data check).

**Result: blocked (data missing).** Sleep before the maze is not in the data set.

## maze-control-sessions

> In long sessions without the maze, SWR amplitude and rate do not differ between the first and second half.

(Results, Multi-region correlations after spatial exploration, paragraph 1)

The data set holds none of the four control sessions (see the data check).

**Result: blocked (data missing).** The control sessions are not in the data set.

## maze-ripple-power-in-spindles

> After maze exploration, CA1 SWR power within spindle cycles increases.

(Results, Multi-region correlations after spatial exploration, paragraph 2)

The data set holds no sleep before the maze (see the data check).

**Result: blocked (data missing).** Sleep before the maze is not in the data set.

## fig7c-run-units-in-spindles

> CA1 units active on the maze are more active in high ripple power spindle cycles than units not active on the maze, with independent effects of activity on the maze and of ripple power.

(Results, Multi-region correlations after spatial exploration, paragraph 2; Figure 7A, C)

In [ ]:
# S3, sleep after the maze. CA1 units are classed by the deposited folders CA1_UnitsInRun and
# CA1_UnitsNotInRun (reading a of fig7c-run-units-in-spindles; readings b and c need the maze spikes,
# which are not in the data set). Cycles: deposited high and low ripple power troughs (dep) or the
# median split of each reading's cycles (spindle-cycles-by-ripple-power).
s3 = S["S3"]
ca1 = [u for u in s3["units"] if u["region"] == "CA1"]
def cycle_sets(rd):
    if rd == "dep":
        d = CYC[("dep", "S3")]
        hi = np.isin(np.round(d.trough.values, 5), np.round(s3["derived"]["SpindleTroughsHiRiPow"], 5))
        return d.assign(hi=hi)
    return CYC[(rd, "S3")]
EV3 = EV["S3"]
rows = []
for rd in READINGS:
    d = cycle_sets(rd)
    cyc = EV3[rd]["cyc"]
    for u in ca1:
        sp = u["spikes"]
        n_in = np.searchsorted(sp, cyc[:, 1]) - np.searchsorted(sp, cyc[:, 0])
        for lab, m in (("high", d.hi.values), ("low", ~d.hi.values)):
            rows.append({"reading": rd, "id": u["id"], "run": u["run"], "ripple": lab, "pct cycles with spikes": 100 * np.mean(n_in[m] > 0),
                         "spikes per cycle": n_in[m].mean()})
RUN = pd.DataFrame(rows)
print(RUN.round(4).to_string(index=False))
res = []
for rd, g in RUN.groupby("reading", sort=False):
    row = {"reading": rd}
    for meas, short in (("spikes per cycle", "spk"), ("pct cycles with spikes", "pct")):
        h = g[g.ripple == "high"]
        row[f"MW p {short} (high cycles)"] = sc.mannwhitney(h[h.run][meas], h[~h.run][meas])
        row[f"run>not {short}"] = h[h.run][meas].mean() > h[~h.run][meas].mean()
        dd = g.rename(columns={meas: "y"}).assign(run=lambda x: x.run.map({True: "run", False: "notrun"}))
        a = sm.stats.anova_lm(ols("y ~ C(run) * C(ripple)", data=dd).fit(), typ=2)
        row[f"ANOVA run p {short}"] = a.loc["C(run)", "PR(>F)"]
        row[f"ANOVA ripple p {short}"] = a.loc["C(ripple)", "PR(>F)"]
        row[f"ANOVA inter p {short}"] = a.loc["C(run):C(ripple)", "PR(>F)"]
    res.append(row)
fig7c = pd.DataFrame(res)
print(fig7c.round(4).T.to_string())
g = RUN[RUN.reading == "dep"].assign(group=lambda x: x.run.map({True: "run", False: "notrun"}) + "-" + x.ripple)
for meas in ("spikes per cycle", "pct cycles with spikes"):
    print(f"Tukey-Kramer, {meas} (deposited cycles):")
    print(pairwise_tukeyhsd(g[meas], g.group).summary())
print("n = 4 run and 4 not-run CA1 units (paper 27 and 15). Smallest two-sided Mann-Whitney p with 4 against 4:", round(2 / 70, 4))

**Result: partially reproduced.** On the 4 run and 4 not-run CA1 units of S3, the two-way ANOVA finds the ripple power effect (p = 0.001 to 0.017), and run units fire more in high ripple power cycles on average under every reading. But neither Mann-Whitney test is significant (p = 0.20 to 0.49 for spikes per cycle, 0.31 to 1.0 for the percent of cycles; 4 against 4 can reach p = 0.029), and the ANOVA run effect is not (p = 0.12 to 0.68).

## maze-ca1-rates-similar

> CA1 units active and not active on the maze do not differ in firing rate in sleep after the maze.

(Results, Multi-region correlations after spatial exploration, paragraph 2)

In [ ]:
rows = []
for u in ca1:
    sp = u["spikes"][sc.in_periods(u["spikes"], s3["sleep"])]
    isi = []
    for a, b in s3["sleep"]:
        x = sp[(sp >= a) & (sp <= b)]
        isi.append(np.diff(x))
    rows.append({"id": u["id"], "run": u["run"], "rate": len(sp) / sc.total_time(s3["sleep"]), "1/mean ISI": 1 / np.concatenate(isi).mean()})
R3 = pd.DataFrame(rows)
print(R3.round(4).to_string(index=False))
for col in ("rate", "1/mean ISI"):
    print(f"{col}: medians run {R3[R3.run][col].median():.4f}, not run {R3[~R3.run][col].median():.4f} spikes/s; Mann-Whitney p = {sc.mannwhitney(R3[R3.run][col], R3[~R3.run][col]):.3f}")

**Result: reproduced.** Run and not-run CA1 units do not differ in sleep firing rate (medians 0.217 against 0.111 spikes/s, Mann-Whitney p = 0.34) or in the reciprocal of the mean inter-spike interval (0.224 against 0.114, p = 0.20), with 4 units per group.

## fig7d-run-units-peri-swr

> CA1 units active on the maze have higher peaks in their peri-SWR histograms (5.09 against 2.34 spikes/s), with no difference between SWRs nested in spindles and others.

(Results, Multi-region correlations after spatial exploration, paragraph 3; Figure 7D left)

In [ ]:
rows = []
for rd in READINGS:
    cpl, unc = NEST[(rd, "S3")]
    allsw = EV3[rd]["swr"]
    for u in ca1:
        c, rate, z, mu, sd = PETH[(u["id"], "swr", rd, "forward")]
        w = sc.window(c, -0.1, 0.1)
        rng = np.random.default_rng(43)
        cc, rc, *_ = sc.peth_z(u["spikes"], cpl, rng, n_shifts=1)
        cu, ru, *_ = sc.peth_z(u["spikes"], unc, rng, n_shifts=1)
        rows.append({"reading": rd, "id": u["id"], "run": u["run"], "peak a": np.nanmax(rate[w]), "peak b": np.nanmax(rate),
                     "nested max": np.nanmax(rc[w]), "other max": np.nanmax(ru[w]), "nested mean": np.nanmean(rc[w]), "other mean": np.nanmean(ru[w])})
F7D = pd.DataFrame(rows)
print(F7D.round(3).to_string(index=False))
summ = []
for rd, g in F7D.groupby("reading", sort=False):
    r_, n_ = g[g.run], g[~g.run]
    row = {"reading": rd}
    for pk in ("a", "b"):
        row[f"run peak {pk}"] = r_[f"peak {pk}"].mean(); row[f"notrun peak {pk}"] = n_[f"peak {pk}"].mean()
        row[f"MW p {pk}"] = sc.mannwhitney(r_[f"peak {pk}"], n_[f"peak {pk}"])
    for grp, gg in (("run", r_), ("notrun", n_)):
        row[f"nested vs other max p ({grp})"] = sc.wilcoxon(gg["nested max"], gg["other max"])
        row[f"nested vs other mean p ({grp})"] = sc.wilcoxon(gg["nested mean"], gg["other mean"])
    summ.append(row)
print(pd.DataFrame(summ).round(3).T.to_string())
print("Expected run 5.09 +- 4.23, not run 2.34 +- 2.09 spikes/s. SWRs: all M-SWR events of S3 (k = 3, or k = 4 in 'results'); "
      "nested and other as in thalamus-at-spindle-nested-swrs. Smallest two-sided Wilcoxon p with 4 units:", sc.min_wilcoxon_p(4))

**Result: partially reproduced.** Run units have higher peri-SWR peaks (2.01 to 2.14 against 0.83 to 0.91 spikes/s, within the paper's 5.09 +- 4.23 and 2.34 +- 2.09) under every reading, but the Mann-Whitney test gives p = 0.057, short of p < 0.05. Nested and other SWRs do not differ in either group (p = 0.125 to 1.0), but with 4 units a Wilcoxon test cannot give less than 0.125.

## fig7b-run-units-spindle-phase

> Spikes of maze-active CA1 units in spindle-nested SWRs are locked to spindle phase (circular mean -1.48 rad); not-run units are not, and Rayleigh Z is higher for run units and for coupled SWRs.

(Results, Multi-region correlations after spatial exploration, paragraph 3; Figure 7B, 7D right)

In [ ]:
# fig7b: spindle phase of CA1 spikes in coupled and uncoupled SWRs. Which spikes: (a) inside the
# above-threshold period of an M-SWR event (k = 3) matched within 20 ms, (b) within +-50 ms of the
# SWR time, (code) within +-75 ms, as the deposited script PhaseLockingCA1UnitsPFCSpindles.mlx.
# Normalisation by the spindle phase distribution of the whole sleep LFP (Methods); the script's
# normalisation (phases of all LFP samples in the same SWR windows) is a variant.
win3 = sc.detect_swr(s3["t"], s3["f"]["ripple"], s3["sleep_mask"], s3["sleep_mask"], 3.0)[1]
pk3 = EV3["methods"]["swr"]
def swr_windows(ev, mode):
    if mode == "b":
        return np.c_[ev - 0.05, ev + 0.05]
    if mode == "code":
        return np.c_[ev - 0.075, ev + 0.075]
    j = np.clip(np.searchsorted(pk3, ev), 1, len(pk3) - 1)
    j = np.where(np.abs(pk3[j] - ev) < np.abs(pk3[j - 1] - ev), j, j - 1)
    ok = np.abs(pk3[j] - ev) < 0.02
    return win3[j[ok]]
rows = []
ph = s3["phase"]["spindle"]
for rd in ("dep", "methods"):
    cpl, unc = NEST[(rd, "S3")]
    for mode in ("a", "b", "code"):
        for lab, ev in (("coupled", cpl), ("uncoupled", unc)):
            W = swr_windows(ev, mode)
            idxw = np.unique(np.concatenate([np.arange(np.searchsorted(s3["t"], a), np.searchsorted(s3["t"], b)) for a, b in W]))
            ref_code = ph[idxw]
            for u in ca1:
                spk = u["spikes"][sc.in_periods(u["spikes"], W)]
                p_ = sc.phase_at(s3["t"], ph, spk)
                r = sc.phase_summary(p_, s3["ref_phase"]["spindle"])
                h2 = sc.normalized_hist(p_, ref_code)
                pc, zc = sc.rayleigh_binned(h2)
                rows.append({"reading": rd, "spikes": mode, "swr": lab, "id": u["id"], "run": u["run"], "n spikes": len(spk),
                             "mean": r["mean_a"], "kappa": r["kappa_a"], "p": r["p_a"], "Z": r["z_a"], "p code norm": pc, "Z code norm": zc})
F7B = pd.DataFrame(rows)
print(F7B[(F7B.reading == "dep") & (F7B.spikes == "code")].round(4).to_string(index=False))
summ = []
for (rd, mode), g in F7B.groupby(["reading", "spikes"], sort=False):
    gc = g[g.swr == "coupled"]
    row = {"reading": rd, "spikes": mode,
           "run coupled circ mean": sc.circ_mean(gc[gc.run]["mean"].values), "run kappa": gc[gc.run].kappa.mean(),
           "run mean p": gc[gc.run].p.mean(), "notrun mean p": gc[~gc.run].p.mean(), "run mean p (code norm)": gc[gc.run]["p code norm"].mean()}
    dd = g.assign(run=g.run.map({True: "run", False: "notrun"}))
    for zc in ("Z", "Z code norm"):
        a = sm.stats.anova_lm(ols(f"Q('{zc}') ~ C(run) * C(swr)", data=dd).fit(), typ=2)
        row[f"ANOVA run p ({zc})"] = a.loc["C(run)", "PR(>F)"]; row[f"ANOVA swr p ({zc})"] = a.loc["C(swr)", "PR(>F)"]
        means = dd.groupby(["run", "swr"])[zc].mean()
        row[f"highest Z group ({zc})"] = "-".join(means.idxmax())
    summ.append(row)
print(pd.DataFrame(summ).round(4).T.to_string())
print("Expected run units in coupled SWRs: circular mean -1.48 +- 1.33 rad, kappa 0.63 +- 0.32, mean Rayleigh p 0.027 +- 0.05; not run 0.218 +- 0.28.")

**Result: partially reproduced.** Run units in coupled SWRs have the highest Rayleigh Z of the four groups under every reading with deposited SWR sets, and under reading (b) their circular mean (-1.37 rad), kappa (0.69) and mean Rayleigh p (0.021) are within the paper's values, with not-run units at mean p 0.33. But the two-way ANOVA finds neither the run effect (p = 0.18 to 0.91) nor the coupling effect (p = 0.055 to 0.99), which the criterion requires.

## fig8-thalamus-unchanged-by-maze

> Thalamic units show no significant change after maze exploration in KC and spindle histograms, spindle phase, or spindle modulation index.

(Results, Multi-region correlations after spatial exploration, paragraph 4; Figure 8A-B)

The data set holds no sleep before the maze (see the data check).

**Result: blocked (data missing).** Sleep before the maze is not in the data set.

## fig8e-ca1-spindle-trend

> CA1 units tend to take part more in high ripple power spindle cycles after maze exploration (60% fire more), without a significant before-after effect; ripple power has a significant effect.

(Results, Multi-region correlations after spatial exploration, paragraph 4; Figure 8C-E)

The data set holds no sleep before the maze (see the data check).

**Result: blocked (data missing).** Sleep before the maze is not in the data set.

## thalamus-not-selective-for-run-cells

> Thalamic firing does not differ between spindle cycles that hold spikes of maze-active CA1 units and those that hold spikes of other CA1 units, nor with cycles in which co-modulated CA1 pairs are active.

(Results, Multi-region correlations after spatial exploration, paragraphs 5 and 6)

In [ ]:
# thalamus-not-selective-for-run-cells, first part: S3 spindle cycles holding spikes of run CA1
# units against cycles holding spikes of not-run CA1 units ((i) any such spike, cycles may be in
# both sets; (ii) only one kind). Per thalamic unit and the thalamic MUA: mean spikes per cycle,
# percent of cycles with a spike, and the peak z in -100..100 ms of the peri-trough histogram (M-PETH).
rows = []
th_streams = [u for u in s3["units"] if u["region"] == "THAL"] + [m for m in s3["mua"] if m["region"] == "THAL"]
for rd in READINGS:
    cyc, tr = EV3[rd]["cyc"], EV3[rd]["sp"]
    has = {}
    for grp in (True, False):
        n = np.zeros(len(cyc), int)
        for u in ca1:
            if u["run"] == grp:
                n += np.searchsorted(u["spikes"], cyc[:, 1]) - np.searchsorted(u["spikes"], cyc[:, 0])
        has[grp] = n > 0
    for sel, (mr, mn) in (("i any", (has[True], has[False])), ("ii only", (has[True] & ~has[False], has[False] & ~has[True]))):
        for u in th_streams:
            row = {"reading": rd, "cycles": sel, "id": u["id"], "n run cycles": int(mr.sum()), "n not-run cycles": int(mn.sum())}
            n_in = np.searchsorted(u["spikes"], cyc[:, 1]) - np.searchsorted(u["spikes"], cyc[:, 0])
            for lab, m in (("run", mr), ("notrun", mn)):
                rng = np.random.default_rng(zlib.crc32(f"{u['id']}|{rd}|{sel}|{lab}".encode()))
                c, r, z, *_ = sc.peth_z(u["spikes"], tr[m], rng)
                row[f"spikes/cycle {lab}"] = n_in[m].mean()
                row[f"% cycles {lab}"] = 100 * np.mean(n_in[m] > 0)
                row[f"peak z {lab}"] = np.nanmax(z[sc.window(c, -0.1, 0.1)])
            rows.append(row)
TSEL = pd.DataFrame(rows)
print(TSEL.round(3).to_string(index=False))
for (rd, sel), g in TSEL[TSEL.id.str.contains("cl-")].groupby(["reading", "cycles"], sort=False):
    print(f"{rd}, {sel}: " + "; ".join(f"{m} run > not-run in {(g[f'{m} run'] > g[f'{m} notrun']).sum()} of {len(g)}, Wilcoxon p {sc.wilcoxon(g[f'{m} run'], g[f'{m} notrun']):.2f}"
                                        for m in ("spikes/cycle", "% cycles", "peak z")))
print("Two thalamic units: smallest two-sided Wilcoxon p", sc.min_wilcoxon_p(2))

**Result: blocked (data missing for the co-modulated pairs; too few units for the rest).** The co-modulated CA1 pairs were chosen from maze activity, which is not in the data set. For the first part, the 2 S3 thalamic units fire more in cycles holding run-unit spikes than in cycles holding not-run spikes with the deposited troughs (2 of 2 units in every measure), and in 1 of 2 with the other troughs; with 2 units the smallest two-sided Wilcoxon p is 0.5 (p = 0.5 to 1.0 here), so the test cannot find a difference.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig1s3-spindles-not-theta | Not reproduced |
| fig2-swr-phase-delta-spindle | Reproduced |
| fig2c-swr-around-kc | Partially reproduced |
| fig3a-thalamic-rate-wake-sleep | Partially reproduced |
| fig3a-bursting-in-sleep | Partially reproduced |
| fig3a-bursting-around-swrs | Partially reproduced |
| swr-shift-control | Partially reproduced |
| fig3c-thalamic-decrease-at-swrs | Not reproduced |
| fig3s1-ca1-mpfc-at-swrs | Partially reproduced |
| glm-thalamus-anticorrelated-swrs | Not reproduced |
| glm-re-versus-other-nuclei | Blocked: metadata insufficient |
| fig3e-thalamic-rebound-after-kc | Partially reproduced |
| kc-trough-timing | Reproduced |
| kc-fraction-of-thalamic-peaks | Partially reproduced |
| kc-ca1-mpfc-symmetric | Partially reproduced |
| kc-regions-compared | Partially reproduced |
| fig3f-asymmetry-index | Reproduced |
| spindle-mua-modulation | Not reproduced |
| fig4-units-not-rhythmic | Not reproduced |
| fig4b-thalamic-pairs-correlated | Not reproduced |
| xcorr-stronger-in-spindles | Partially reproduced |
| fig5c-spindle-phase-locking | Partially reproduced |
| fig5c-delta-phase-locking | Partially reproduced |
| phase-robust-to-spike-count | Reproduced (the test cannot reject at n = 5) |
| rebound-delta-phase-trend | Reproduced (r near 0) |
| fig5d-phase-lags-consistent | Blocked: other (no uncorrelated pair) |
| spindle-cycles-by-ripple-power | Partially reproduced |
| fig6-modulation-with-ripple-power | Not reproduced |
| modulation-amplitude-versus-ripple | Partially reproduced |
| fig6e-quartile-synergy | Blocked: other (too few units); a change to not reproduced was proposed and not agreed |
| fig6f-quartile-versus-median | Partially reproduced |
| fig6g-spike-count-unchanged | Reproduced (the tests cannot reject at n = 3) |
| xcorr-pairs-ripple-modulation | Blocked: other (one pair) |
| thalamus-at-spindle-nested-swrs | Partially reproduced |
| mpfc-units-coupled-swrs | Reproduced |
| mua-coupled-swrs | Partially reproduced |
| maze-swr-kc-increase | Blocked: data missing |
| maze-control-sessions | Blocked: data missing |
| maze-ripple-power-in-spindles | Blocked: data missing |
| fig7c-run-units-in-spindles | Partially reproduced |
| maze-ca1-rates-similar | Reproduced |
| fig7d-run-units-peri-swr | Partially reproduced |
| fig7b-run-units-spindle-phase | Partially reproduced |
| fig8-thalamus-unchanged-by-maze | Blocked: data missing |
| fig8e-ca1-spindle-trend | Blocked: data missing |
| thalamus-not-selective-for-run-cells | Blocked: data missing (maze spikes for the pairs; two units for the rest) |

Of 46 claims, 8 are reproduced, 21 partially reproduced, 7 not reproduced and 10 blocked: 6 for data
missing from the data set (the maze claims, and the run-cell claim, whose co-modulated pairs need
maze spikes), 1 for metadata, and 3 for other reasons (no uncorrelated thalamic pair; one pair;
fig6e-quartile-synergy, whose tests 3 units cannot reach and whose proposed change to not reproduced
the third verification round did not agree with). With 5 thalamic units a
two-sided Wilcoxon test cannot give p below 0.0625, and with 3 units 0.25, so the many partial outcomes
are mostly effects that go the paper's way on a few units without the test the criterion asks for.

How the analysis differs from the paper's: The data set holds 3 of the paper's sessions and
5 of its 63 thalamic units, so every unit-level result rests on a few units, and proportions can
take only a few values. Following the claims file, values and directions are compared on this
subset. The deposited spindle troughs do not follow the Methods' 3 sd rule, so every analysis is run
on the deposited events and on events detected by the Methods' and the Results' rules; outcomes are
judged under any reading the claims file allows. S3 deposits no full list of SWRs, so its SWRs are
detected here (M-SWR) in every reading, and its deposited and Methods readings share them. Waking is
all time outside the sleep periods (no position data). The sleep periods are the deposited ones; the
Methods' sleep rule as written (1 s and 10 s averages) finds almost no sleep, which is shown for
fig3a with a variant that fills short dips. Phase normalisation and the Rayleigh test on binned
weighted angles follow the deposited script `PhaseLockingCA1UnitsPFCSpindles.mlx`. The
trough-triggered scalograms of fig1s3 are shown in raw power and relative to the state's mean power.

Changes after the first verification round: the autocorrelogram peak finder now takes the peak after
the first minimum that follows lag 0, under three smoothings, and is checked on a synthetic 10 Hz
process; the modulation index is taken from a histogram smoothed over +-1 s and then cropped, with a
flat-firing reference; the Rayleigh test of M-phase reading (a) no longer applies a bin-spacing
correction, as the deposited script; most claims blocked only because a test or tolerance band
cannot be met at this n are now tested on their values and directions, as the claims file's unit
samples paragraph says; the quantities of fig6, its quartile claims, the pair correlogram index, the
fig6g tests and the run-cell thalamic comparison are computed; the in-cycle pair correlograms have a
cycle-permutation control; the SWR shift control uses 20 draws per SWR; the M-sleep readings (b) and
(c) are run for fig3a; and the deposited scripts' relevant lines are quoted in the data check.

What was found about the data set: It holds three sessions (about 3.5 h), not the 32 sessions
with thalamic units the paper analysed, and none of the maze epoch, sleep before the maze, position
data or control sessions. The thalamic nucleus of units is not given, and Session 2's units are
placed in no region by their folder. The deposited spindle troughs are 3.7 to 6.2 times as many as
the Methods' rule gives and reach -0.7 sd, and the deposited S1 SWRs are more than the Methods'
k = 3 rule gives. The Methods' sleep rule, applied as written, does not reproduce the deposited sleep
periods. Spike times are stored at single precision (0.98 ms resolution in S3), which makes repeated
timestamps. In S3 the mPFC unit and MUA include 24 min before the LFP and every other stream begin,
which the descriptor does not explain.

Changes after the second verification round: the three fig6 claims still blocked for too few units
are rated on their values and directions, as the other claims at this n are, and changes of outcome are
proposed for a third round: modulation-amplitude-versus-ripple and fig6f-quartile-versus-median to
partially reproduced, fig6e-quartile-synergy to not reproduced. Their outcomes stay blocked until that
round. Wording on fig3f and fig4 is corrected, and the GLM prints its Wald p values in full.

Changes after the third verification round: modulation-amplitude-versus-ripple and
fig6f-quartile-versus-median are partially reproduced, as proposed and agreed. The proposed change of
fig6e-quartile-synergy to not reproduced was not agreed (the verifier would rate it partially
reproduced), so it stays blocked with both views recorded. The conclusions now say that the similarity
tests of modulation-amplitude-versus-ripple carry no information at n = 3, that the ripple power spread
also widens in fig6f, and that the three claims use spindle amplitude reading (a) only.